# 거래 건수 기반 법인고객 위축 신호 탐지
## 종합 분석 보고서

> 거래 건수가 직전 6개월보다 35% 이상 줄어든 법인은, 이후 6개월간 요구불입금액이 크게 약화되는 비율이 같은 조건의 비교 법인보다 약 4.7배 높았음.

---

### 읽는 순서

| 장 | 내용 | 대응 노트북 |
|---|---|---|
| 1 | 문제 정의 — 무엇을 알고 싶은가 | – |
| 2 | 데이터 — 어떤 법인을, 어떤 변수로 | 01·02 |
| 3 | 채널 구조 — 왜 5개 채널을 합쳐서 보는가 | 03 |
| 4 | 위축 신호 정의 — 규칙을 어떻게 정했나 | 04 |
| 5 | 통계 기법 맛보기 — 층화·표준화, 부트스트랩, 평균 회귀 (합성 예시) | – |
| 6 | 핵심 결과 — 신호와 요구불입금액 약화의 연관 | 05 |
| 7 | 고객별 차이 — 규모·채널 유형·자동이체·채널 이동 | 05·06 |
| 8 | 분석 선택을 바꿔도 같은가 — 민감도 분석 20가지 | 07 |
| 9 | 보강 점검 — 얼마나 맞히고 놓치나, 기준 기간 반론 | 08 |
| 10 | 활용 방안 — 신호가 잡히면 무엇을 하나 | 06~08 |
| 11 | 결론·한계·후속 과제, 용어 사전 | – |

### 그림을 읽는 방법

- **파랑** = 건수 신호가 잡힌 기업(주인공), **회색** = 비교 기업·기준, **주황** = 결과(요구불입금액)·강조·기준선.
- 숲 그림(forest plot)에서 **점 = 추정값, 선 = 95% 신뢰구간**, 회색 점선 = RR 1(연관 없음).

In [ ]:
# ------------------------------------------------------------------
# 0. 준비: 라이브러리 · 한글 글꼴 · 색 · 그림 도우미 함수
#    (원본 파일은 아래 '0. 실행'에서 읽습니다. numpy · pandas · matplotlib 만 필요)
# ------------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from matplotlib.patches import Rectangle, FancyBboxPatch, FancyArrowPatch

# 한글 글꼴: 설치된 것 중 첫 번째를 사용 (Windows 맑은 고딕 / Mac AppleGothic / Linux 나눔고딕)
_installed = {f.name for f in fm.fontManager.ttflist}
FONT = next((f for f in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"] if f in _installed), "DejaVu Sans")

mpl.rcParams.update({
    "font.family": FONT, "axes.unicode_minus": False,
    "figure.dpi": 100, "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#9AA3AF", "axes.linewidth": 0.8,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": "#E6E9EE", "grid.linewidth": 0.7,
    "axes.titlesize": 12.5, "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.titlepad": 10,
    "axes.labelsize": 10, "axes.labelcolor": "#444B55", "xtick.labelsize": 9.5, "ytick.labelsize": 9.5,
    "legend.frameon": False, "legend.fontsize": 9.5,
})

# 색 약속: 파랑 = 신호 기업(주인공), 회색 = 비교 기업/기준, 주황 = 결과(요구불입금액)·강조·기준선, 보라 = 보조 비교
BLUE, ORANGE, PURPLE = "#1F5FAE", "#C4600A", "#7A5195"
GRAY, LIGHT, INK, MUTED = "#8A93A0", "#D9DEE5", "#222222", "#5F6773"
PALE_BLUE, PALE_ORANGE = "#BCD2EE", "#F0CBA8"


def src(fig, text):
    """그림 아래에 출처·주의 문구를 작게 적는다."""
    fig.text(0.012, 0.004, text, fontsize=8.2, color=MUTED, ha="left", va="bottom")


def finish(fig, text=None, rect=(0, 0.04, 1, 1)):
    fig.tight_layout(rect=rect)
    if text:
        src(fig, text)
    plt.show()


def fmt_ci(est, lo, hi, d=2):
    return f"{est:.{d}f} ({lo:.{d}f}~{hi:.{d}f})"


def forest(ax, rows, xmax=None, ref=1.0, base=None, left_pad=0.0, label_fs=9.5, show_text=True, xlabel="상대위험도(RR)  ·  점 = 추정값, 선 = 95% 신뢰구간"):
    """숲 그림(forest plot): rows = [(이름, 추정값, 하한, 상한, 색), ...], 위에서 아래로 그린다.
    점선 = RR 1(연관 없음), 주황 점선 = base(비교 기준값). xmax 를 넘는 신뢰구간은 잘라서 ▶ 로 표시한다."""
    n = len(rows)
    xmax = xmax or max(r[3] for r in rows) * 1.15
    for i, (lab, est, lo, hi, col) in enumerate(rows):
        y = n - 1 - i
        ax.plot([lo, min(hi, xmax)], [y, y], color=col, lw=2.2, solid_capstyle="round", zorder=2)
        if hi > xmax:
            ax.plot([xmax], [y], marker=">", color=col, ms=6, zorder=3, clip_on=False)
        ax.plot([est], [y], "o", color=col, ms=8, mec="white", mew=1.2, zorder=3)
        if show_text:
            ax.text(xmax * 1.015, y, fmt_ci(est, lo, hi), va="center", ha="left", fontsize=9, color=INK, clip_on=False)
    ax.axvline(ref, color=GRAY, ls="--", lw=1.1, zorder=1)
    if base is not None:
        ax.axvline(base, color=ORANGE, ls=(0, (4, 3)), lw=1.1, zorder=1)
    ax.set_yticks(range(n))
    ax.set_yticklabels([r[0] for r in rows][::-1], fontsize=label_fs)
    ax.set_ylim(-0.7, n - 0.3)
    ax.set_xlim(0 - left_pad, xmax)
    ax.grid(axis="y", visible=False)
    ax.set_xlabel(xlabel)
    return ax

---
# 0. 실행 — 원본 파일에서 집계 결과까지

이 노트북은 **원본 파일을 넣고 직접 돌리는 보고서**임. 위에서부터 실행하면 아래 순서로 진행됨.

1. **설정**(0.1): 원본 경로 등을 정함.
2. **계산**(0.2): 노트북 `01`~`08`의 코드를 같은 코드·같은 난수 시드로 실행해 집계 결과를 저장함. 끝난 단계는 건너뜀.
3. **점검**(0.3): 이번 계산이 기준 실행(2026-10-02)과 같은지 점검표로 확인함.
4. **보고서**(1~11장): 저장된 집계 결과로 통계 용어 설명과 그림 40개·표를 그림. 값은 미리 적어 둔 상수가 아니라 **이번 실행에서 계산된 값**임.

- **걸리는 시간**: 처음부터 전부 계산하면 엑셀 읽기와 부트스트랩(03 10,000회, 05~08 2,000회) 때문에 **수십 분**이 걸릴 수 있음. 01~08을 이미 돌려 둔 상태이면 끝난 단계는 건너뛰어 **몇십 초**면 끝남(설정의 `FORCE_RERUN`).
- **사용한 값은 두 종류**임. **집계 결과**(01~08이 저장한 요약값, 법인 식별 정보·행 단위 값 없음, 5 미만 칸은 가림, 그림 아래에 `집계 결과 · 노트북 0n`)와, 5장의 **합성(가짜) 예시**(통계 기법을 설명하려고 만든 가짜 데이터, 그림 아래에 `합성 예시`, 실제 결과가 아님).
- **본문 설명 속 숫자**(예: "RR 4.69")는 기준 실행(2026-10-02, 실제 데이터)의 값을 적어 둔 문장임. 다른 원본이나 다른 설정으로 돌리면 그림·표의 값과 문장 속 숫자가 달라질 수 있으므로 0.3의 **점검표**를 먼저 확인함.
- **데이터 취급**: 원본 경로는 레포 밖에 둠. 실행하면 출력 셀에 집계값과 그림이 저장되고 설정 셀에 로컬 경로가 남으므로, 레포에 올리기 전에 출력과 경로를 정리하고 팀의 데이터 반출 검토를 거침.

### 0.1 설정

In [ ]:
# ------------------------------------------------------------------
# 설정 — 여기만 고치세요
# ------------------------------------------------------------------
import os, sys, json, time
from pathlib import Path

# 원본 파일(레포 밖에 두세요). 경로를 코드 밖으로 내보내지 마세요.
DATA_PATH = os.environ.get("CNT_DATA_PATH", r"여기에_원본_경로.xlsx")
SHEET = os.environ.get("CNT_SHEET", "all")                   # 시트가 하나면 "0"
WORK_DIR = os.environ.get("CNT_WORK_DIR") or None            # 기본값: 레포의 data/work_cnt (.gitignore 대상)

FORCE_RERUN = False   # True  = 01~08을 원본으로 처음부터 다시 계산 (수십 분)
                      # False = 이미 계산된 단계는 건너뜀 (결과 파일이 있을 때) -> 아래 보고서만 새로 그림
BOOT = None           # 부트스트랩 횟수. None = 노트북 기본값(03: 10,000 / 05~08: 2,000) -> 기준 실행과 같은 값
REAL_DATA = os.environ.get("CNT_NO_EXPECT") is None   # 실제 원본이면 True: 01의 행·법인 수 기대값 점검을 켬 (합성 데이터면 False)

def _find_lib():
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base / "code", base / "건수분석" / "code", base / "yangdaecheon" / "건수분석" / "code"):
            if (cand / "cntlib.py").exists():
                return cand
    raise SystemExit("cntlib.py 를 찾지 못했습니다. 노트북을 yangdaecheon/건수분석/notebooks 에서 여세요.")

sys.path.insert(0, str(_find_lib()))
import cntlib as cl
import report_cnt as rc
WORK = cl.resolve_work_dir(WORK_DIR)
print("산출물 폴더:", WORK)
print("원본 파일:", "(설정됨)" if DATA_PATH else "(없음)", "| 처음부터 다시 계산:", FORCE_RERUN)

### 0.2 계산 실행 — 노트북 01~08

| 단계 | 하는 일 | 저장하는 집계 결과 |
|---|---|---|
| 01 | 원본 읽기 → 법인 × 36개월 패널, 표본(P36·P35만·P35+) 정의, 빈 달 처리 시험 | `nb01_summary.json`, `output/nb01/` |
| 02 | 구간 코드 → 대표 건수, 총거래활동량 I1, 상한 구간 민감도 | `quant_spec.json`, `output/nb02/` |
| 03 | 채널 간 상관·이동·주성분분석 | `nb03_summary.json`, `output/nb03/` |
| 04 | 위축 신호 규칙(비교 기준·k·활동량 하한·감소 기준 n) | `detect_spec_v2.json`, `output/nb04/` |
| 05 | 요구불입금액 검증(RR·부트스트랩·독립 표본·경로) | `nb05_summary.json`, `output/nb05/` |
| 06 | 채널 유형·자동이체·채널 이동 | `nb06_summary.json`, `output/nb06/` |
| 07 | 민감도 분석 20가지 | `nb07_summary.json`, `output/nb07/` |
| 08 | 적중·포착·기업 단위 시점, 기준 기간 비공유 신호 | `nb08_summary.json`, `output/nb08/` |

> 각 단계는 해당 노트북의 **코드 셀을 그대로 이어 붙여** 실행함(같은 코드, 같은 난수 시드). 그래서 이미 01~08을 돌린 값과 같은 결과가 나옴. 법인ID는 읽자마자 내부 번호로 바꾸고 어디에도 남기지 않으며, 화면에는 집계값만 나옴.
> 단계가 실패하면 그 단계에서 멈추고 로그 위치와 에러 종류·줄 번호만 보여 줌(데이터 값이 섞인 메시지는 공유하지 않음).

In [ ]:
# 노트북 01~08의 코드를 순서대로 실행한다 (화면에는 단계 이름과 시간만 출력, 상세 출력은 WORK/logs 에 저장).
# 05 의 35개월 법인 결과는 P36 분석·기록을 마친 뒤의 종합본이므로 열어서 실행한다.
t0 = time.time()
status = rc.run_pipeline(DATA_PATH, WORK, sheet=SHEET, boot=BOOT, force=FORCE_RERUN, expect_real_counts=REAL_DATA)
print(f"\n파이프라인 끝: {time.time() - t0:.0f}초  (로그: {WORK / 'logs'})")

### 0.3 결과 불러오기와 점검표

계산이 끝난 집계 결과를 읽고, **기준 실행(2026-10-02, 실제 데이터)의 핵심 값과 비교한 점검표**를 만듦. 같은 원본·같은 코드·같은 시드이면 전부 `일치`여야 함.

In [ ]:
# 01~08이 저장한 '집계 결과'(CSV·JSON)만 읽는다. 행 단위 파일은 열지 않는다.
D = rc.load_results(WORK)
globals().update(D)                      # 이후 그림·표가 쓰는 값(MAIN, SAMPLE_RR, VARIANTS ...)
if SYNTHETIC_RUN:
    print(cl.SYNTHETIC_NOTICE)
if LOAD_WARNINGS:
    print("불러오기 경고 (표가 비었거나 열이 달라 일부 값을 비워 둠):", "; ".join(LOAD_WARNINGS))

chk = rc.checkpoints(D)
display(chk.set_index("점검"))
n_diff = int((chk["판정"] == "차이").sum())
if n_diff == 0:
    print("모든 점검 값이 기준 실행(2026-10-02)과 일치함 -> 본문 설명의 숫자를 그대로 믿고 읽어도 됨.")
else:
    print(f"{n_diff}개 값이 기준 실행과 다름 -> 합성 데이터이거나 다른 원본·설정이면 정상. "
          "같은 원본인데 다르면 01의 검증표와 설정(BOOT, FORCE_RERUN)을 확인하세요. 본문 설명 속 숫자는 기준 실행 값임.")

### 결과 한눈에 보기

자세한 근거와 통계 기법 설명은 아래 1~11장에 있음. 아래 카드는 그 결과를 한 장으로 모은 것임.

In [ ]:
def card(ax, x, y, w, h, big, label, cap, color=BLUE):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.08", facecolor="white", edgecolor=LIGHT, lw=1.6))
    ax.add_patch(Rectangle((x, y + 0.08), 0.07, h - 0.16, facecolor=color, edgecolor="none"))
    ax.text(x + 0.22, y + h - 0.32, label, fontsize=10.5, color=MUTED, va="center", fontweight="bold")
    ax.text(x + 0.22, y + h / 2 - 0.02, big, fontsize=22, color=color, va="center", fontweight="bold")
    ax.text(x + 0.22, y + 0.34, cap, fontsize=9.3, color=INK, va="center", linespacing=1.4)

vmin = min(v[2] for v in VARIANTS); vmax = max(v[2] for v in VARIANTS)
cards = [
    (f"RR {MAIN['rr']:.2f}", "상대위험도 (36개월 법인)", f"95% 신뢰구간 {MAIN['lo']:.2f} ~ {MAIN['hi']:.2f}\n신호 기업의 약화가 비교 기업의 약 {MAIN['rr']:.1f}배", BLUE),
    (f"{MAIN['p_det']}% vs {MAIN['p_ctrl']}%", "요구불입금액 약화 비율", f"신호 기업 {MAIN['n1']}곳 중 {MAIN['obs']}곳 vs\n같은 구성의 비교 기업", BLUE),
    (f"RR {SAMPLE_RR[1][2]:.2f}", "독립 검증 표본 (35개월 법인만)", f"규칙을 정하지 않은 {N_SAMPLE['P35만']:,}곳, 신호 {SAMPLE_RR[1][1]}곳\n95% 신뢰구간 {SAMPLE_RR[1][3]:.2f} ~ {SAMPLE_RR[1][4]:.2f}", PURPLE),
    (f"{vmin:.2f} ~ {vmax:.2f}", "분석 조건 20가지를 바꿔도", "모두 신뢰구간 하한 > 1\n(연관이 사라지는 선택 없음)", PURPLE),
    (f"적중 {HIT_P36['precision']}% · 포착 {HIT_P36['recall']}%", "선별 도구의 성격", "신호 기업의 약 75%는 약화가 아니고\n약화의 약 89%는 신호 없이 일어남", ORANGE),
    (f"RR {YOY['P36_yoy']['rr']:.2f}", "기준 기간 비공유 신호", f"전년 같은 달 대비로 다시 정의해도 유지\n(독립 표본 {YOY['P35_yoy']['rr']:.2f})", ORANGE),
]
fig, ax = plt.subplots(figsize=(13.2, 5.0))
ax.set_xlim(0, 13.2); ax.set_ylim(0, 5.0); ax.axis("off"); ax.grid(False)
for i, (big, lab, cap, col) in enumerate(cards):
    r_, c_ = divmod(i, 3)
    card(ax, 0.1 + c_ * 4.4, 2.6 - r_ * 2.5, 4.1, 2.25, big, lab, cap, col)
finish(fig, "집계 결과 · 노트북 05·07·08 · 자세한 근거는 6~9장", rect=(0, 0.03, 1, 1))

---
# 1장. 문제 정의 — 무엇을 알고 싶은가

### 1.1 배경

- 법인고객은 급여이체·자금관리·여신 약정이 묶여 있어 거래 은행을 바꾸기 어렵고, 계좌 해지에도 인감·등기 서류가 필요함. 그래서 **계좌를 닫지 않고 거래만 줄이는** 경우가 많고, 이런 고객은 계좌 해지 기준의 이탈률에 잡히지 않음.
- 팀은 이탈을 **계좌 해지가 아닌 거래 위축**으로 다시 정의했음. 이 분석은 그중 **거래 건수(채널 이용 횟수)만으로 위축 신호를 만들 수 있는지**를 다룸.
- 역할 분담(2026-10-01 팀 회의): **건수로 신호를 만들고**, 요구불입금액은 신호를 만드는 데 쓰지 않은 채 **신호가 맞았는지 확인하는 결과 지표**로만 사용함. (요구불입금액으로 위축을 판단하는 분석은 팀 내 별도로 진행)

> **쉽게 말해** 거래 횟수가 눈에 띄게 줄어든 법인을 먼저 골라 두고, 그 법인들의 이후 요구불입금액이 정말 더 많이 줄었는지 '정답 확인용'으로 맞춰 보는 구조임. 신호를 만들 때 정답(요구불입금액)을 미리 보지 않아야 확인이 공정해지기 때문임.

### 1.2 핵심 질문

1. 건수 감소는 이후 **요구불입금액 약화**와 연결되는가?
2. 어떤 고객에서 강하고 어떤 고객에서 약한가?
3. 분석 선택(가정·기준·표본)을 바꿔도 결론이 같은가?
4. 신호가 잡히면 어떻게 활용할 수 있는가?

### 1.3 시간 구조 한눈에 보기(사용, 다듬어서)
아래 그림은 이 분석의 시간 구조임. **신호가 처음 잡힌 달을 기준월 e**로 부르고, 그 달을 0으로 두었음.

- **건수(신호)**: 최근 6개월 평균 `R`을 직전 6개월 평균 `C`와 비교해 변화율 `r = (R − C) / C`를 구함. `r ≤ −35%`이면 신호.
- **요구불입금액(결과)**: 신호 후 6개월 평균을 **전년 같은 달** 평균과 비교해 변화율 `g`를 구함. `g`가 아주 낮으면(하위 5%, −64.2% 이하) **약화**.
- 눈여겨볼 점: 신호의 비교 기간(`C`)과 약화의 기준 기간(전년 동기)이 **같은 달**임(그림의 회색 구간). 이 점이 9장에서 점검할 '기준 기간 공유' 반론의 출발점임.

In [ ]:
def window(ax, y, a, b, color, label, hatch=None, text_color="white", h=0.62, fs=9.5, alpha=1.0):
    """a~b (포함) 달 구간을 가로 막대로 그린다."""
    ax.add_patch(Rectangle((a - 0.5, y - h / 2), b - a + 1, h, facecolor=color, edgecolor="white", lw=1.2, hatch=hatch, alpha=alpha, zorder=2))
    ax.text((a + b) / 2, y, label, ha="center", va="center", fontsize=fs, color=text_color, zorder=3, fontweight="bold")

fig, ax = plt.subplots(figsize=(11.5, 4.4))
ax.set_xlim(-17.8, 6.8); ax.set_ylim(-0.55, 3.75)
ax.grid(False); ax.spines["left"].set_visible(False); ax.set_yticks([])
ax.set_xticks(range(-17, 7)); ax.set_xticklabels([f"{m:+d}" if m else "e" for m in range(-17, 7)], fontsize=8.5)
ax.set_xlabel("기준월 e(신호가 처음 잡힌 달)를 0으로 둔 상대 월")

# 건수(신호) 줄: 비교 기간 [e-11, e-6], 최근 6개월 [e-5, e]
ax.text(-17.6, 3.2, "건수 (신호를 만드는 쪽)", fontsize=10.5, color=BLUE, fontweight="bold", va="center")
window(ax, 2.55, -11, -6, GRAY, "비교 기간 (직전 6개월)  C")
window(ax, 2.55, -5, 0, BLUE, "최근 6개월  R")
ax.text(-5.3, 2.0, "변화율 r = (R - C) ÷ C 가 -35% 이하이면  →  신호", fontsize=10, color=BLUE, va="center", fontweight="bold")

# 요구불입금액(결과) 줄: 전년 동기 [e-11, e-6], 신호 후 6개월 [e+1, e+6]
ax.text(-17.6, 1.2, "요구불입금액 (결과를 확인하는 쪽)", fontsize=10.5, color=ORANGE, fontweight="bold", va="center")
window(ax, 0.55, -11, -6, GRAY, "전년 동기 (기준)", hatch="///", text_color=INK, alpha=0.55)
window(ax, 0.55, 1, 6, ORANGE, "신호 후 6개월")
ax.text(-5.3, 0.0, "변화율 g = (신호 후 - 전년 동기) ÷ 전년 동기 가 -64.2% 이하이면  →  약화", fontsize=10, color=ORANGE, va="center", fontweight="bold")

# 같은 달 연결선
ax.plot([-11.5, -11.5], [0.86, 2.24], color=GRAY, ls=":", lw=1.2); ax.plot([-5.5, -5.5], [0.86, 2.24], color=GRAY, ls=":", lw=1.2)
ax.text(-8.5, 1.5, "같은 달 (기준 기간 공유)", ha="center", va="center", fontsize=9.5, color=MUTED, bbox=dict(boxstyle="round,pad=0.25", fc="white", ec=GRAY, lw=0.8))
ax.axvline(0.5, color=INK, lw=1.2); ax.text(0.7, 3.55, "기준월 e: 여기까지 알고 신호를 판정", ha="left", fontsize=9.5, color=INK)
ax.set_title("건수로 신호를 만들고, 그 이후의 요구불입금액으로 확인한다")
finish(fig, "개념도 · 실제 분석 설정(비교 기간 k=6개월, 감소 기준 35%, 약화 기준 -64.2%)을 그린 것임")

---
# 2장. 데이터 — 어떤 법인을, 어떤 변수로 (노트북 01·02)

### 2.1 분석 대상 법인: 세 가지 표본(사용)

| 표본 | 정의 | 역할 |
|---|---|---|
| **P36** | 2023.01~2025.12 **36개월 모두 관측**된 법인 | **규칙 결정 표본** — 신호 규칙의 기준값을 정하고 주 결과를 내는 표본 |
| **P35만** | **정확히 35개월** 관측된 법인(P36과 겹치지 않음) | **독립 검증 표본** — 규칙을 정하는 데 쓰지 않고 같은 규칙을 그대로 적용 |
| **P35+** | 35개월 이상 = P36 + P35만 | **표본 확대** — 표본이 늘어 불확실성이 줄지만 P36을 포함하므로 독립 증거는 아님 |

> **통계 용어 — 홀드아웃(hold-out) 검증, 독립 검증**
> 규칙(예: 35% 감소)과 기준값을 정할 때 쓴 데이터에서 그 규칙의 성능을 재면, 규칙이 그 데이터에 **맞춰져 성능이 좋게 나올 수 있음**(낙관 편향).
> 그래서 규칙을 정하는 데 쓰지 않은 다른 데이터(홀드아웃)에 **같은 규칙을 그대로** 적용해 결과가 재현되는지 확인함.
> **쉽게 말해** 시험 문제를 만든 반 학생들로만 시험을 보면 점수가 후하게 나올 수 있으니, 문제를 만들 때 몰랐던 다른 반에서도 비슷한 점수가 나오는지 보는 것임.

34개월 이하 관측된 법인은 빈 달이 2개 이상이라 이번 분석에서 제외했음(전체 법인 15,473곳 중 4,784곳을 분석).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.1), gridspec_kw={"width_ratios": [1.15, 1]})

# (왼쪽) 표본 규모
ax = axes[0]
labs = ["전체 법인", "35개월 이상 (P35+)", "36개월 완전관측 (P36)", "35개월만 관측 (P35만)"]
vals = [N_ALL, N_SAMPLE["P35+"], N_SAMPLE["P36"], N_SAMPLE["P35만"]]
cols = [LIGHT, PALE_BLUE, BLUE, PURPLE]
bars = ax.barh(range(4), vals, color=cols, height=0.62)
for i, v in enumerate(vals):
    ax.text(v + 150, i, f"{v:,}곳 ({v / N_ALL * 100:.1f}%)", va="center", fontsize=10, color=INK)
ax.set_yticks(range(4)); ax.set_yticklabels(labs); ax.invert_yaxis(); ax.set_xlim(0, 21000)
ax.set_xlabel("법인 수 (곳)"); ax.grid(axis="y", visible=False)
ax.set_title("분석에 쓴 법인은 전체의 약 31%")

# (오른쪽) 관측 개월 수 분포
ax = axes[1]
months = [int(m) for m in OBS_MONTHS]
counts = [OBS_MONTHS[str(m)] for m in months]
cc = [BLUE if m == 36 else PURPLE if m == 35 else LIGHT for m in months]
ax.bar(months, counts, color=cc, width=0.72)
for m, c in zip(months, counts):
    ax.text(m, c + 50, f"{c:,}", ha="center", fontsize=9.5, color=INK)
ax.set_xticks(months); ax.set_xlabel("관측된 개월 수 (36개월 중)"); ax.set_ylabel("법인 수 (곳)")
ax.set_ylim(0, 3900); ax.grid(axis="x", visible=False)
ax.set_title("관측 개월 수별 법인 수 (30~36개월만 표시)")
finish(fig, "집계 결과 · 노트북 01 (samples, obs_months_30_36)")

### 2.2 건수 변수와 '구간 코드'(사용)

- 분석에 쓴 건수 변수는 8개(창구·인터넷뱅킹·스마트뱅킹·폰뱅킹·ATM·자동이체·외환 수출·외환 수입 거래건수)이고, 금액 변수 3개(요구불 입금·출금·예금잔액)는 **신호를 만드는 데는 쓰지 않고** 결과 확인용으로 따로 읽었음.
- **건수는 정확한 횟수가 아니라 10개 구간 코드(0~9)** 로만 제공됨. 예를 들어 코드 5는 '11~20건', 코드 9는 '50건 초과'임.
- 국내 5채널(창구·인터넷·스마트·폰·ATM)을 합쳐 **총거래활동량 I1**(= 5채널 근사 건수의 합)을 만들 것이므로, 구간 코드를 **대표 건수**로 바꿔야 함.

> **통계 용어 — 순서형(ordinal) 자료와 구간 중앙값**
> 구간 코드는 '크기 순서'만 있는 자료임(코드 6이 코드 5보다 크다). 합계를 내려면 각 구간을 **하나의 대표 값**(여기서는 구간의 중앙값)으로 바꿔야 하고, 이는 **가정**임.
> 가장 큰 구간(50건 초과)은 위쪽이 열려 있어 중앙값을 구할 수 없으므로, **60건으로 가정**했음(사전에 고정한 값, 근거 자료 없음).
> 그래서 이 가정이 결과를 좌우하는지 **민감도 분석**(8장)으로 확인했음 — 51건·100건으로 바꾸거나, 구간 하한·상한으로 합산하거나, 구간 번호 자체를 더해 보아도 결론이 유지됨.

In [ ]:
labels = ["0", "1", "2", "3~5", "6~10", "11~20", "21~30", "31~40", "41~50", "50 초과"]
mids = [0, 1, 2, 4, 8, 15.5, 25.5, 35.5, 45.5, 60]
fig, ax = plt.subplots(figsize=(10.5, 3.9))
cols = [BLUE] * 9 + [ORANGE]
bars = ax.bar(range(10), mids, color=cols, width=0.7)
bars[9].set_hatch("///"); bars[9].set_facecolor(PALE_ORANGE); bars[9].set_edgecolor(ORANGE)
for i, m in enumerate(mids):
    ax.text(i, m + 1.2, f"{m:g}", ha="center", fontsize=10, color=INK)
ax.annotate("열린 구간 → 60건으로 '가정'\n(민감도 분석: 51·100건 등)", xy=(9, 61), xytext=(6.3, 56), fontsize=9.5, color=ORANGE,
            arrowprops=dict(arrowstyle="->", color=ORANGE), va="center")
ax.set_xticks(range(10))
ax.set_xticklabels([f"코드 {i}\n({l}건)" for i, l in enumerate(labels)], fontsize=8.8)
ax.set_ylabel("대표 건수 (건/월)"); ax.set_ylim(0, 70); ax.grid(axis="x", visible=False)
ax.set_title("구간 코드를 대표 건수로 바꾼다 — 구간 폭이 고르지 않음")
finish(fig, "집계 결과 · 노트북 02 (bands) · 대표 건수 = 구간의 중앙값, 코드 9는 가정값")

### 2.3 채널별 이용 현황과 '상한 구간' 문제(안씀)

아래 그림은 36개월 법인의 기업-월(법인×월) 중 **해당 채널을 쓴 비율**과 **상한 구간(50건 초과)에 있는 비율**임.

- 인터넷뱅킹은 거의 모든 법인이 쓰고(81.5%), 쓰는 법인의 상당수가 **월 50건을 넘김**(전체 기업-월의 30.9%).
- 상한 구간에 있는 기업은 건수가 늘거나 줄어도 **구간 코드가 그대로**라 변화가 둔하게 잡힘 → 거래가 아주 많은 기업은 건수 신호로 감지하기 어려운 **구조적 한계**가 됨(7장에서 확인).
- 외환(수출·수입)은 기업-월 기준 이용 비율이 2% 이하(이용 경험이 있는 기업도 36개월 법인의 약 8%)이고 제조업·도소매업에 편중되어 있어 국내 채널 신호에 넣지 않고 **별도 축**으로 둠. 자동이체는 정기 지출 성격(공과금·임대료·상환 등)이라 줄어도 위축이 아닐 수 있어 역시 **별도 축**으로 둠.

In [ ]:
chs = ["인터넷", "자동이체", "스마트", "ATM", "창구", "폰", "외환수입", "외환수출"]
use = [USAGE_P36[c][0] for c in chs]; ceil = [USAGE_P36[c][1] for c in chs]
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.1), sharey=True)
for ax, vals, title, col in ((axes[0], use, "해당 채널을 쓴 기업-월 비율(%)", BLUE), (axes[1], ceil, "상한 구간(50건 초과) 비율(%)", ORANGE)):
    ax.barh(range(len(chs)), vals, color=col, height=0.62)
    for i, v in enumerate(vals):
        ax.text(v + 1.2, i, f"{v:.1f}", va="center", fontsize=9.5)
    ax.set_title(title); ax.set_xlim(0, 112); ax.grid(axis="y", visible=False)
axes[0].set_yticks(range(len(chs))); axes[0].set_yticklabels(chs); axes[0].invert_yaxis()
axes[0].set_xlabel("%"); axes[1].set_xlabel("%")
finish(fig, "집계 결과 · 노트북 02 (usage_ceiling, 36개월 법인 기업-월 121,392건)")

# 국내 5채널 중 상한 구간 채널이 몇 개인가
fig, ax = plt.subplots(figsize=(9.5, 1.9))
left = 0
for (k, v), c in zip(CEIL_CHANNELS.items(), [LIGHT, PALE_ORANGE, ORANGE]):
    ax.barh(0, v, left=left, color=c, height=0.5, edgecolor="white")
    ax.text(left + v / 2, 0, f"{k}\n{v:.1f}%", ha="center", va="center", fontsize=9.5, color=INK)
    left += v
ax.set_xlim(0, 100); ax.set_yticks([]); ax.grid(False); ax.spines["left"].set_visible(False)
ax.set_xlabel("기업-월 중 비율(%)"); ax.set_title("한 달에 상한 구간에 있는 국내 채널 수")
finish(fig, "집계 결과 · 노트북 02 (i1_cap_sensitivity)", rect=(0, 0.08, 1, 1))

### 2.4 빈 달 처리 — 채우지 않는다(안씀)

- 35개월 법인은 한 달이 **기록 없음**임. (1,412곳 중 중간 달이 비는 경우가 89.4%)
- 빈 달은 **거래가 없어서 기록이 생성되지 않은 달일 수 있음**(해석 가설). 임의로 채우면 실제보다 활동이 많은 것으로 가정하게 됨.
- 기본 처리(**M0, 쌍 맞춤**): 빈 달을 **채우지 않고**, 두 시점을 비교할 때 **두 시점 모두 관측된 달끼리만** 짝지어 평균함. 짝이 부족하면 그 시점은 '판정 불가'로 두고 정상으로 세지 않음.
- 민감도(**M1**): 같은 기업의 앞뒤 관측값으로 채워서 다시 계산함. 이를 쓰려면 정확도가 충분해야 하므로, 36개월 법인의 **관측된 달 하나를 일부러 가렸다가 채워서 맞히는 시험**을 했음.

> **통계 용어 — 결측(missing)과 보간(interpolation)**
> 결측은 값이 없는 것이고, 보간은 앞뒤 값으로 빈 곳을 채우는 것임. 보간은 편리하지만 '없던 값'을 만들어 내므로, 결측의 이유를 모를 때는 **채우지 않는 쪽을 기본**으로 하고 채운 결과를 민감도로 확인하는 것이 안전함.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.0), gridspec_kw={"width_ratios": [0.8, 1.2]})
ax = axes[0]
keys = list(MISSING_KIND); vals = [MISSING_KIND[k][1] for k in keys]; cnt = [MISSING_KIND[k][0] for k in keys]
ax.bar(range(3), vals, color=[GRAY, BLUE, GRAY], width=0.62)
for i, (v, c) in enumerate(zip(vals, cnt)):
    ax.text(i, v + 2, f"{v:.1f}%\n({c:,}곳)", ha="center", fontsize=10)
ax.set_xticks(range(3)); ax.set_xticklabels(["첫 달\n(2023.01)", "중간 달", "마지막 달\n(2025.12)"])
ax.set_ylim(0, 105); ax.set_ylabel("35개월 법인 1,412곳 중 비율(%)"); ax.grid(axis="x", visible=False)
ax.set_title("빈 달의 위치")

ax = axes[1]
vs = ["폰", "외환수출", "외환수입", "스마트", "창구", "ATM", "인터넷", "자동이체"]
ex = [FILL_BY_VAR[v][0] for v in vs]; w1 = [FILL_BY_VAR[v][1] for v in vs]
x = np.arange(len(vs)); w = 0.38
ax.bar(x - w / 2, ex, w, color=PALE_BLUE, label="정확히 일치")
ax.bar(x + w / 2, w1, w, color=BLUE, label="±1 구간 이내")
ax.axhline(90, color=ORANGE, ls="--", lw=1.1, label="채택 기준 90%")
ax.set_xticks(x); ax.set_xticklabels(vs, fontsize=9.2); ax.set_ylim(40, 108); ax.set_ylabel("맞힌 비율(%)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3); ax.grid(axis="x", visible=False)
ax.set_title(f"가렸다 채워 맞히기: 전체 정확 {FILL_ALL['exact']}% · ±1 구간 이내 {FILL_ALL['within1']}%")
finish(fig, "집계 결과 · 노트북 01 (p35_missing_kind, fill_accuracy_by_var) · 전체로는 기준(90%)을 넘지만 인터넷·ATM 등 일부 변수는 못 미쳐 채움(M1)은 기본이 아닌 민감도로만 사용")

**정리**

- 분석 대상 = **P36 3,372곳(규칙 결정)** + **P35만 1,412곳(독립 검증)**. 빈 달은 채우지 않음.
- 구간 코드는 대표 건수로 바꾸되 **50건 초과 = 60건은 가정**임(민감도 분석으로 확인).
- 데이터 품질 점검(행 수 365,988, 고유 법인 15,473, (법인·월) 중복 0건, 금액 변환 실패·음수 0건)은 모두 통과했음.

---
# 3장. 채널 구조 — 왜 5개 채널을 합쳐서 보는가 (왼쪽만 씀))

신호를 **채널별로 따로** 만들지, **합쳐서 하나로** 만들지 결정하려고 세 가지를 확인했음.

1. 채널 사이에 **함께 움직이는 구조**가 있는가? (상관·주성분분석)
2. 한 채널이 줄 때 다른 채널이 늘어나는 **채널 이동**이 흔한가? (이동이 흔하면 개별 채널 감소는 '위축'이 아닌 '이동'일 수 있음)
3. 월 단위 채널 변동은 **잡음**이 큰가?

> **통계 용어 — 상관계수(correlation)**
> 두 변수가 함께 움직이는 정도(−1~+1). +1이면 항상 같이 늘고 줄며, 0이면 관련이 없음.
> 여기서는 두 가지로 나누어 봄 — **기업 간(between)**: '인터넷을 많이 쓰는 기업은 창구도 많이 쓰는가'(기업의 평소 수준), **기업 내(within)**: '이 기업이 이번 달 평소보다 인터넷을 더 쓴 달에 창구도 더 썼는가'(월별 변동).
> 기업 규모가 큰 기업은 모든 채널을 많이 쓰므로 기업 간 상관은 높게 나오기 쉽지만, 위축 신호에 필요한 것은 기업 **내** 월별 변동임.

In [ ]:
ch = CH8
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.2))
for ax, M, title in ((axes[0], CORR_BETWEEN, "기업 간 상관: 평소 수준이 비슷하게 움직임"), (axes[1], CORR_WITHIN, "기업 내 상관: 월별 변동은 거의 독립")):
    A = np.array(M, dtype=float)
    im = ax.imshow(A, cmap=mpl.colors.LinearSegmentedColormap.from_list("w2b", ["#FFFFFF", BLUE]), vmin=0, vmax=0.7)
    for i in range(8):
        for j in range(8):
            ax.text(j, i, f"{A[i, j]:.2f}", ha="center", va="center", fontsize=8.6, color="white" if A[i, j] > 0.38 else INK)
    ax.set_xticks(range(8)); ax.set_xticklabels(ch, rotation=40, ha="right", fontsize=9)
    ax.set_yticks(range(8)); ax.set_yticklabels(ch, fontsize=9); ax.grid(False)
    ax.set_title(title, fontsize=11.5)
fig.subplots_adjust(left=0.07, right=0.90, bottom=0.2, top=0.92, wspace=0.28)
cax = fig.add_axes([0.92, 0.22, 0.014, 0.66])
fig.colorbar(im, cax=cax, label="상관계수 (0~0.7 색 범위)")
src(fig, "집계 결과 · 노트북 03 (between_corr_P36, within_corr_P36) · 36개월 법인 · 두 그림 모두 같은 색 범위")
plt.show()

**읽는 법**: 왼쪽은 칸이 짙고 오른쪽은 거의 흰색임. 기업 간에는 국내 채널·자동이체끼리 평소 수준이 같이 움직이지만(0.37~0.68), 한 기업의 **월별 변동**은 채널끼리 거의 독립임(−0.02~+0.12).

- 즉 채널 하나하나의 월 단위 변화에는 **채널 고유의 잡음**이 많이 섞여 있음 → 단일 채널 감소는 신호로 쓰기에 불안정함.
- 외환은 ATM과만 약하게 함께 움직임(0.10~0.12).(이거 안씀)

In [ ]:
labs = TRANSITION_COLS
T = np.array([[np.nan if v is None else v for v in TRANSITION[r]] for r in labs], dtype=float)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3), gridspec_kw={"width_ratios": [1.05, 1]})
ax = axes[0]
Z = np.where(np.isnan(T), 0, T)
ax.imshow(np.log10(Z + 1), cmap=mpl.colors.LinearSegmentedColormap.from_list("w2b", ["#FFFFFF", BLUE]), vmin=0, vmax=3.6)
for i in range(4):
    for j in range(4):
        v = T[i, j]
        txt = "<5" if np.isnan(v) else f"{int(v):,}"
        ax.text(j, i, txt, ha="center", va="center", fontsize=10.5, color="white" if (not np.isnan(v) and v > 300) else INK, fontweight="bold" if i == j else None)
ax.set_xticks(range(4)); ax.set_xticklabels(labs); ax.set_yticks(range(4)); ax.set_yticklabels(labs); ax.grid(False)
ax.set_xlabel("2025년 주 온라인 채널"); ax.set_ylabel("2023년 주 온라인 채널")
ax.set_title("주 온라인 채널은 2년 사이 거의 바뀌지 않음")

ax = axes[1]
n_net = int(T[0, 1]); back = int(T[1, 0]); tot = int(np.nansum(T[0]))
vals = [T[0, 0] / np.nansum(T[0]) * 100, T[0, 1] / np.nansum(T[0]) * 100]
ax.barh([1, 0], [vals[0], vals[1]], color=[BLUE, ORANGE], height=0.55)
ax.text(vals[0] + 1.5, 1, f"{vals[0]:.1f}%  (인터넷 → 인터넷)", va="center", fontsize=10)
ax.text(vals[1] + 1.5, 0, f"{vals[1]:.1f}%  (인터넷 → 스마트)", va="center", fontsize=10)
ax.set_yticks([1, 0]); ax.set_yticklabels(["그대로", "스마트로\n이동"]); ax.set_xlim(0, 135); ax.grid(axis="y", visible=False)
ax.set_xlabel("2023년 인터넷 주 이용 기업 중 비율(%)")
ax.set_title("인터넷에서 스마트로 옮겨 간 기업은 드묾")
finish(fig, "집계 결과 · 노트북 03 (transition_P36) · 5 미만 칸은 가려져 있음('<5') · 주 온라인 채널 = 인터넷·스마트·폰 중 연평균 근사 건수가 가장 큰 채널(모두 0이면 '없음')")
net = abs(int(T[0, 1]) - int(T[1, 0]))
print(f"인터넷에서 스마트로 옮겨 간 기업 {int(T[0,1])}곳, 스마트에서 인터넷으로 옮겨 간 기업 {int(T[1,0])}곳 → 서로 상쇄되어 순이동은 {net}곳 (36개월 법인 {N_SAMPLE['P36']:,}곳의 약 {net / N_SAMPLE['P36'] * 100:.1f}%)")

> **통계 용어 — lift(리프트)와 신뢰구간으로 본 '동시 발생'**
> '인터넷이 줄어든 달'과 '스마트가 늘어난 달'이 **우연히 겹칠 때 기대되는 빈도**보다 몇 배 자주 같이 일어나는지를 lift라고 함. 1이면 우연 수준, 1보다 크면 같이 일어나는 경향임.
> 채널 이동이 흔하다면 '인터넷↓ & 스마트↑'가 우연보다 훨씬 자주 같이 나타나야 함.

(안씀)

In [ ]:
rows = [("인터넷↓·스마트↑\n(반대 방향, 확증 검정)", NB03["lift_opp_internet_down_smart_up_P36"], 0.83, 1.08, BLUE),
        ("인터넷↓·스마트↓\n(같은 방향)", NB03["lift_same_internet_down_smart_down_P36"], 1.12, 1.41, GRAY),
        ("[보조] 총활동 유지 창 한정\n(해석에 쓰지 않음)", NB03["lift_opp_stable_internet_down_smart_up_P36"], 2.45, 3.43, PURPLE)]
fig, ax = plt.subplots(figsize=(10.5, 3.5))
forest(ax, rows, xmax=4.0, ref=1.0, xlabel="lift  ·  점 = 추정값, 선 = 95% 신뢰구간 (1 = 우연 수준)")
ax.set_title("'인터넷이 줄 때 스마트가 는다'는 채널 이동은 확인되지 않음")
fig.subplots_adjust(right=0.80)
finish(fig, "집계 결과 · 노트북 03 (lift_P36) · 보조 행은 총 활동이 유지된 창만 봐서 인위적으로 커질 수 있어 판정에서 제외", rect=(0, 0.04, 0.80, 1))

**읽는 법**: 확증 검정(반대 방향 동시 발생)의 lift가 0.95로 1과 구별되지 않음 → **채널 이동 가설은 지지되지 않았음**. 오히려 같은 방향(둘 다 줄거나 늘)의 lift가 1.26으로 약간 높아 '같이 줄어드는' 경향이 있음.

### 주성분분석(PCA)으로 본 채널 구조

> **통계 용어 — 주성분분석(PCA)과 고유값**
> 여러 변수가 **공통의 큰 흐름 몇 개**로 요약되는지 보는 방법임. 각 주성분이 설명하는 비율이 크게 차이 나면 변수들 사이에 뚜렷한 구조가 있다는 뜻이고, **모두 비슷하면 구조 없이 제각각 움직인다**는 뜻임. 고유값이 1 근처이면 '독립 잡음 하나'와 같은 크기임.

(안씀)

In [ ]:
fig, ax = plt.subplots(figsize=(8.8, 3.7))
x = np.arange(1, 9)
ax.bar(x, PCA_VAR, color=[BLUE if v > 12.5 else LIGHT for v in PCA_VAR], width=0.62)
for i, v in zip(x, PCA_VAR):
    ax.text(i, v + 0.3, f"{v:.1f}", ha="center", fontsize=9.5)
ax.axhline(12.5, color=ORANGE, ls="--", lw=1.1); ax.text(8.45, 16.6, "점선 = 8개 변수가 완전히 독립일 때 각 12.5%", ha="right", color=ORANGE, fontsize=9.5)
ax.set_xticks(x); ax.set_xticklabels([f"PC{i}" for i in x]); ax.set_ylim(0, 19); ax.set_ylabel("설명하는 분산 비율(%)"); ax.grid(axis="x", visible=False)
ax.set_title("월별 변동은 뚜렷한 공통 구조 없이 제각각 (설명 비율이 모두 10~15%)")
finish(fig, "집계 결과 · 노트북 03 (pca_variance) · 기업·월 평균을 제거한 월별 변동의 주성분 분석(8개 건수 변수)")

### 3장의 결론 — 왜 합산 지수 하나로 가는가

| 확인 | 결과 | 설계에의 영향 |
|---|---|---|
| 기업 내 월별 채널 변동의 상관 | −0.02~+0.12 (거의 독립 잡음) | 채널 하나의 월 단위 감소는 **잡음이 큼** → 합산·평균으로 잡음을 줄임 |
| 인터넷→스마트 채널 이동 | 확증 lift 0.95로 지지되지 않음, 2년간 주 채널 유지 약 95% | 채널 이동이 신호를 흔들 가능성이 작음. 합산하면 이동은 대부분 **상쇄**됨 |
| 주성분분석 | 모든 성분이 10~15%로 비슷 | 채널별로 따로 볼 만한 뚜렷한 구조가 없음 |

→ **국내 5채널(창구·인터넷·스마트·폰·ATM)의 근사 건수 합 I1을 신호의 재료**로 쓰고, **자동이체·외환은 별도 축**으로 둠.
탐색적으로는 '창구↓→스마트↑', 'ATM↓→스마트↑·폰↑'처럼 반대 방향 동시 발생 lift가 1을 넘는 쌍이 일부 있었으나(1.3~1.8) 보고만 하고 설계에는 쓰지 않았음.

---
# 4장. 위축 신호 정의 — 규칙을 어떻게 정했나 (노트북 04)

신호 규칙은 아래 네 가지를 정하는 일임. **모두 요구불입금액을 보지 않고, 건수 자료의 분포만으로** 정했음(결과를 보고 규칙을 고치면 '정답에 맞춘 규칙'이 되어 검증이 무의미해지기 때문).

| 결정 | 질문 | 이 분석의 선택 |
|---|---|---|
| ① 무엇을 볼까 | 신호의 재료가 되는 지수 | 국내 5채널 근사 건수 합 **I1**(총거래활동량) |
| ② 무엇과 비교할까 | 직전 기간 대비? 전년 같은 달 대비? 몇 개월 창? | **직전 기간 대비, k = 6개월** |
| ③ 어떤 기업은 판정하지 않을까 | 거래가 너무 적은 기업은 변화율이 우연히 크게 흔들림 | **활동량 하한 Φ = 11.17**(월평균 근사 건수) 미만은 판정 제외 |
| ④ 얼마나 줄면 신호인가 | 감소 기준 n | **35%** (변화율 하위 5% 지점을 반올림) |

### 4.1 변화율은 어떻게 계산하나 — 한 기업 예시

기준월 e에서 **최근 6개월 평균 R**을 **직전 6개월 평균 C**와 비교해 변화율을 구함.

$$ r = \frac{R - C}{C}, \qquad R = \text{최근 6개월}[e-5,\,e]\ \text{평균}, \quad C = \text{직전 6개월}[e-11,\,e-6]\ \text{평균} $$

- **판정 가능** = 비교 평균 `C`가 활동량 하한 `Φ` 이상이고, 두 시점이 함께 관측된 달이 충분함.
- **신호** = 판정 가능한 달 중 `r`이 −35% 이하로 **처음** 내려간 달(기업당 최초 1회). 판정 대상 기준월은 **2024.06~2025.06의 13개월**임(신호 후 6개월을 추적해야 하므로 2025.06까지).
- 아래는 **합성 예시**임(실제 기업이 아님). 월 평균 50건을 쓰던 가짜 기업이 거래를 줄이는 상황을 만들어 규칙이 어떻게 작동하는지 보여 줌. 이 셀의 `toy_change`가 실제 코드(`code/detect_cnt.py`의 `window_change`·`first_detect`)를 줄여서 옮긴 것임.

(씀, 더 이쁘게)

In [ ]:
# ---- 합성 예시: 한 기업의 월별 총거래활동량 I1 (건/월). 실제 기업이 아님 ----
rng = np.random.default_rng(11)
t = np.arange(36)
level = 50 - 30 * np.clip((t - 19) / 6, 0, 1)          # 20번째 달부터 6개월에 걸쳐 50건 → 20건으로 감소
I = np.maximum(level + rng.normal(0, 3, 36), 0)

K, PHI, N_PCT = 6, 11.17, 0.35

def toy_change(I, e, k=K):
    """기준월 e 에서 최근 k개월 평균 R, 직전 k개월 평균 C, 변화율 r = (R - C) / C"""
    R = I[e - k + 1: e + 1].mean()
    C = I[e - 2 * k + 1: e - k + 1].mean()
    return R, C, (R - C) / C

E_RANGE = range(17, 30)                                  # 판정 가능한 기준월(2024.06~2025.06에 해당)
rows = []
for e in E_RANGE:
    R, C, r = toy_change(I, e)
    rows.append((e, R, C, r, (C >= PHI) and (r <= -N_PCT)))   # 판정 가능(C >= 하한) 이면서 r <= -35% 이면 신호 후보
df = pd.DataFrame(rows, columns=["기준월 e", "최근 평균 R", "직전 평균 C", "변화율 r", "신호 조건 충족"])
e_sig = int(df.loc[df["신호 조건 충족"], "기준월 e"].iloc[0])             # 기업당 최초 1회
print(f"이 예시 기업의 신호 기준월: e = {e_sig}  (변화율 r = {df.set_index('기준월 e').loc[e_sig, '변화율 r']:.1%})")

ym = [f"{2023 + m // 12}.{m % 12 + 1:02d}" for m in range(36)]
fig, axes = plt.subplots(2, 1, figsize=(11.5, 6.4), sharex=True, gridspec_kw={"height_ratios": [1.5, 1]})
ax = axes[0]
ax.axvspan(e_sig - 11.5, e_sig - 5.5, color=LIGHT, alpha=0.9, zorder=0); ax.axvspan(e_sig - 5.5, e_sig + 0.5, color=PALE_BLUE, alpha=0.8, zorder=0)
ax.plot(t, I, color=BLUE, lw=2, marker="o", ms=4)
R, C, r = toy_change(I, e_sig)
ax.hlines(C, e_sig - 11.5, e_sig - 5.5, color=GRAY, lw=2.4); ax.hlines(R, e_sig - 5.5, e_sig + 0.5, color=BLUE, lw=2.4)
ax.text(e_sig - 8.5, 60.5, f"직전 6개월\n평균 C = {C:.1f}건", ha="center", va="top", fontsize=9.8, color=MUTED, fontweight="bold")
ax.text(e_sig - 2.5, 60.5, f"최근 6개월\n평균 R = {R:.1f}건", ha="center", va="top", fontsize=9.8, color=BLUE, fontweight="bold")
ax.axvline(e_sig + 0.5, color=INK, lw=1.2); ax.text(e_sig + 0.8, 60.5, f"신호 기준월 e\n({ym[e_sig]})", fontsize=9.8, va="top")
ax.set_ylim(5, 62); ax.set_ylabel("총거래활동량 I1 (건/월)"); ax.set_title("합성 기업의 월별 거래 건수와 신호 판정 창")

ax = axes[1]
es = df["기준월 e"].to_numpy(); rs = df["변화율 r"].to_numpy() * 100
ax.plot(es, rs, color=BLUE, lw=2, marker="o", ms=5)
ax.axhline(-35, color=ORANGE, ls="--", lw=1.3); ax.text(17.1, -31, "감소 기준: -35%", color=ORANGE, fontsize=9.8, va="bottom")
ax.plot([e_sig], [r * 100], "o", color=ORANGE, ms=11, mec="white", mew=1.5, zorder=4)
ax.annotate("처음으로 -35% 이하 → 신호", xy=(e_sig, r * 100), xytext=(e_sig - 6.2, -53), fontsize=9.8, color=ORANGE, arrowprops=dict(arrowstyle="->", color=ORANGE))
ax.set_ylabel("변화율 r (%)"); ax.set_ylim(-62, 12)
ax.set_xticks(range(0, 36, 3)); ax.set_xticklabels([ym[m] for m in range(0, 36, 3)])
ax.set_xlabel("기준월 (2023.01 = 0번째 달)"); ax.set_title("기준월마다 구한 변화율 (판정 가능한 13개월만)")
finish(fig, "합성 예시 · 실제 기업이 아님. 규칙의 작동 방식을 설명하기 위한 가짜 자료")

### 4.2 비교 기준과 창 길이 k — '일시적 하락'을 피한다

**비교 기준**: 거래 건수에 해마다 반복되는 계절 패턴이 있다면 '전년 같은 달'과 비교해야 계절 효과가 사라짐. 그런데 노트북 04의 계절성 점검에서 **계절성이 없다고 판정되어**(사전에 정한 점검 규칙 적용) 더 최근 정보를 쓰는 **직전 기간 대비**로 정했음.

**창 길이 k**: 비교 창이 너무 짧으면 '잠깐 줄었다가 곧 회복하는' 경우가 신호로 잡힘. 그런 경우를 **일시적 하락**이라 하고, 후보 1·2·3·6개월 중 일시적 하락 비율이 가장 낮은 창을 고름.

> **통계 용어 — 일시적 하락 비율**
> 신호가 잡힌 기업 가운데, 신호 이후 k개월 동안 건수가 전년 같은 달 대비 −10% 이내로 **다시 회복한 기업의 비율**임. 이 비율이 낮을수록 신호가 '일시적 흔들림'이 아닌 '지속적 감소'를 더 잘 잡는다는 뜻임.
> 사전 선택 규칙: 비율이 20% 미만인 가장 짧은 창. 모두 20% 이상이면 비율이 더 이상 줄지 않는 창(인접 창 비율 차이의 z검정)을 고름.

(안씀)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.4))
ks = [1, 2, 3, 6]
for name, col, mk in (("P36 (규칙 결정)", BLUE, "o"), ("P35만 (독립 검증)", PURPLE, "s"), ("P35+ (표본 확대)", GRAY, "^")):
    key = name.split(" ")[0]
    ys = [v for _, v in TRANSIENT_30[key]]
    ax.plot(ks, ys, color=col, marker=mk, lw=2, ms=7, label=name)
    ax.text(6.12, {"P36": 22.6, "P35만": 29.8, "P35+": 26.2}[key], f"{ys[-1]:.1f}%", va="center", fontsize=9.5, color=col)
ys35 = [v for _, v in TRANSIENT_35_P36]
ax.plot(ks, ys35, color=BLUE, ls=":", lw=1.8, marker="o", ms=4, label="P36, 최종 감소 기준 35%로 다시 계산")
ax.axhline(20, color=ORANGE, ls="--", lw=1.2); ax.text(1.0, 21.2, "사전 기준 20%: 이 아래인 창이 있으면 가장 짧은 창을 선택", color=ORANGE, fontsize=9.3)
ax.set_xticks(ks); ax.set_xticklabels([f"{k}개월" for k in ks]); ax.set_xlim(0.8, 6.7)
ax.set_ylim(0, 64); ax.set_ylabel("일시적 하락 비율(%)"); ax.set_xlabel("비교 창 길이 k")
ax.legend(loc="upper right", fontsize=9)
ax.set_title("창이 길수록 '잠깐 줄었다 회복'하는 사례가 줄어든다 → k = 6개월")
finish(fig, "집계 결과 · 노트북 04 (transient_share, 임시 감소 기준 30%) · 노트북 07 (selection_shares, 최종 기준 35%)")

**읽는 법**: 모든 표본에서 일시적 하락 비율이 창이 길어질수록 줄어듦(P36: 1개월 57% → 6개월 25%). 20% 미만인 창은 없었으므로 규칙에 따라 **비율이 계속 줄어드는 가장 긴 창 6개월**을 선택했음. 최종 감소 기준(35%)으로 다시 계산해도 같은 결론임(3개월 42.8%, 6개월 25.4%).

- **한계**: 6개월 창에서도 신호의 약 1/4은 회복 사례가 섞여 있음. 그래서 신호 기업의 상당수가 약화로 이어지지 않는 것이 자연스러움(6장·9장).
- 후보를 6개월까지로 둔 이유: 비교 창이 길수록 판정 가능한 달이 줄어듦(후보월 수 = 19 − k, 36개월 자료).

### 4.3 활동량 하한 Φ — 작은 기업의 우연한 출렁임을 거른다

거래가 월 1~2건인 기업은 건수가 한두 건만 바뀌어도 변화율이 −50%, +100%처럼 크게 흔들림. 그런 기업까지 신호로 보면 대부분 **우연**임. 그래서 직전 6개월 평균 활동량 `C`가 **하한 Φ 미만이면 판정하지 않음**.

> **통계 용어 — 사분위범위(IQR)**
> 값을 작은 순서로 세웠을 때 25% 지점과 75% 지점의 차이로, 값이 얼마나 **흩어져 있는지**를 나타냄. 이상치에 덜 민감함.
> 하한 Φ는 '활동량이 이 이상이면 변화율의 흩어짐(IQR)이 충분히 안정되는 값'으로 정함.

아래 그림은 활동량 5분위별 변화율의 IQR임. 활동량이 작을수록 IQR이 커서(변화율이 크게 흔들려서) 판정이 불안정함.

(안씀)

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 4.3))
x = np.arange(5); w = 0.38
a = IQR_BY_ACT["천장 채널 없음"]; b = IQR_BY_ACT["천장 채널 있음"]
ax.bar(x - w / 2, [v[1] for v in a], w, color=BLUE, label="상한 구간(50건 초과) 채널이 없는 칸")
ax.bar(x + w / 2, [v[1] for v in b], w, color=PALE_ORANGE, edgecolor=ORANGE, hatch="///", label="상한 구간 채널이 있는 칸")
for i, v in enumerate(a):
    ax.text(i - w / 2, v[1] + 0.04, f"{v[1]:.2f}", ha="center", fontsize=9.3, color=BLUE)
for i, v in enumerate(b):
    ax.text(i + w / 2, v[1] + 0.04, f"{v[1]:.2f}", ha="center", fontsize=9.3, color=ORANGE)
ax.set_xticks(x); ax.set_xticklabels(["1분위\n(작음)", "2분위", "3분위", "4분위", "5분위\n(큼)"])
ax.set_xlabel("직전 6개월 평균 활동량 C의 5분위 (두 집단을 각각 5등분)"); ax.set_ylabel("변화율의 사분위범위(IQR)"); ax.set_ylim(0, 2.3)
ax.legend(loc="upper right"); ax.grid(axis="x", visible=False)
ax.set_title("활동량이 작을수록 변화율이 크게 흔들린다")
finish(fig, "집계 결과 · 노트북 04 (diag_ceiling_iqr) · 36개월 법인")

tab = pd.DataFrame({"분위": [f"{i + 1}분위" for i in range(5)],
                    "[상한 구간 채널 없음] C 범위(건/월)": [v[0] for v in a], "IQR": [v[1] for v in a],
                    "[상한 구간 채널 있음] C 범위(건/월)": [v[0] for v in b], "IQR ": [v[1] for v in b]})
display(tab.round(2).set_index("분위"))

**읽는 법**: 파란 막대(정상적인 칸)는 활동량이 커질수록 IQR이 2.0 → 0.2로 **계속 줄어듦**. 이 흩어짐이 안정되는 지점이 하한 후보임.

> **규칙을 한 번 고친 이야기 (v1 → v2)** — 처음 등록한 규칙(v1)대로 계산하자 하한이 **상한 구간의 가정값(60)** 근처로 정해졌음. 주황 빗금 막대처럼 **상한 구간(50건 초과)에 있는 채널이 있는 칸은 건수가 가정값에 '기계적으로 고정'**되어 변화율이 인위적으로 작게(IQR 0.04~0.07) 나오기 때문이었음. 그 결과 판정 가능 기업의 97%가 상한 구간 보유 기업이고, 가정값을 51·60·100으로 바꿀 때마다 신호 기업이 크게 달라졌음(겹침 0.5 정도).
> 이를 **요구불입금액을 보기 전에** 건수 분포만으로 진단해서, 하한을 정할 때 **상한 구간 채널이 있는 칸을 빼고** 계산하는 수정 규칙(v2)을 도입했음(변경 이력에 '결과를 본 뒤 변경' 여부와 함께 기록). 원래 규칙(v1)은 8장의 민감도 분석에 포함했고 결과는 RR 5.60으로 같은 방향이었음.
> 그 결과 **Φ = 11.17**(월평균 근사 건수)이 되었음.

### 4.4 감소 기준 n — 백분위수로 정한다

> **통계 용어 — 백분위수(percentile)**
> 값을 작은 것부터 줄 세웠을 때 **하위 5% 지점의 값**을 '5번째 백분위수'라고 함. '극단적인 상태'를 **분포의 꼬리**로 정의하는 방법임.
> 장점은 기준 숫자를 연구자가 임의로 고르지 않고 **자료의 분포에서 얻는다**는 점임(다만 '5%'와 반올림 단위는 연구자의 선택임). 단점은 5%라는 비율 자체가 관례적이어서 업무상 손실과 직접 이어지지 않는다는 점임.

판정 가능한 기업-월의 변화율을 줄 세운 뒤 **하위 5% 지점(−34.6%)을 5%p 단위로 반올림**해 **감소 기준 35%**로 정했음.

(씀)

In [ ]:
# ---- 합성 예시: 변화율 분포의 모양을 흉내 낸 가짜 자료. 실제 분포가 아님 ----
rng = np.random.default_rng(5)
sim = np.concatenate([rng.normal(-0.01, 0.08, 9000), -rng.gamma(2.0, 0.12, 1000)])
sim *= abs(RULE["q05"]) / 100 / abs(np.percentile(sim, 5))       # 5번째 백분위수가 실제 값(-34.6%)이 되도록 눈금만 조정
q05 = np.percentile(sim, 5)
fig, ax = plt.subplots(figsize=(10.5, 4.2))
bins = np.linspace(-1.0, 0.5, 90)
cnt, edges, patches = ax.hist(sim, bins=bins, color=LIGHT)
for p_, left in zip(patches, edges[:-1]):
    if left + (edges[1] - edges[0]) <= q05:
        p_.set_facecolor(ORANGE)
ax.axvline(q05, color=ORANGE, lw=1.6)
ax.text(q05 - 0.01, cnt.max() * 0.88, f"하위 5% 지점\n{q05 * 100:.1f}%", ha="right", color=ORANGE, fontsize=10.5, fontweight="bold")
ax.annotate("→ 5%p 단위로 반올림:\n   감소 기준 35%", xy=(-0.35, cnt.max() * 0.35), xytext=(-0.28, cnt.max() * 0.62), fontsize=10, color=INK, arrowprops=dict(arrowstyle="->", color=INK))
ax.set_xlabel("변화율 r (직전 6개월 대비)"); ax.set_ylabel("기업-월 수 (상대적 크기)"); ax.set_yticks([])
ax.xaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0, decimals=0)); ax.grid(axis="x", visible=False)
ax.set_title("백분위수 문턱: 변화율이 가장 낮은 5%를 '신호'의 경계로 삼는다")
finish(fig, "합성 예시 · 분포의 모양은 설명용(가짜 자료). 실제 분석 값은 하위 5% 지점 -34.6% 하나뿐임")

**이 기준이 안정적인가?** 기준월을 연도별로 나눠 같은 방식으로 구해도 **2024년 34.8%, 2025년 34.4%**(전체 34.6%)로 거의 같았음. 또한 25%·45%로 바꿔 다시 계산한 결과도 연관이 유지되었음(8장).

### 4.5 판정 가능한 기업과 신호 기업

활동량 하한 Φ를 적용하면 36개월 법인 3,372곳 중 **2,511곳(74.5%)이 판정 가능**하고, 그중 **488곳(19.4%)에서 신호**가 잡힘.

(씀)

In [ ]:
v = V2_COUNTS["P36"]
tot, judge, sig = N_SAMPLE["P36"], v["judgeable"], v["detected"]
below = FLOOR_EXCLUDED["n"]; undef = tot - judge - below    # 활동량 하한 미만 / 그 밖(변화율 정의 불가 등)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.0), gridspec_kw={"width_ratios": [1.25, 1]})
ax = axes[0]
ax.barh(2, tot, color=LIGHT, height=0.55); ax.barh(1, judge, color=PALE_BLUE, height=0.55); ax.barh(0, sig, color=BLUE, height=0.55)
ax.text(tot + 40, 2, f"{tot:,}곳  36개월 법인", va="center", fontsize=10.2)
ax.text(judge + 40, 1, f"{judge:,}곳  판정 가능 ({v['judge_pct']}%)", va="center", fontsize=10.2)
ax.text(sig + 40, 0, f"{sig}곳  신호 기업 (판정 가능의 {v['rate_pct']}%)", va="center", fontsize=10.2)
ax.text(judge + 40, 1.42, f"제외 {tot - judge}곳 = 활동량 하한 미만 {below} + 그 밖 {undef}", fontsize=9.3, color=MUTED, va="center")
ax.set_yticks([]); ax.set_xlim(0, 5400); ax.grid(False); ax.spines["left"].set_visible(False); ax.set_xlabel("법인 수 (곳)")
ax.set_title("판정 가능 기업과 신호 기업")

ax = axes[1]
ub = np.array(UNITS_BY_MONTH)
lab = [f"{str(int(m))[2:4]}.{str(int(m))[4:]}" for m in ub[:, 0]]
ax.bar(range(len(ub)), ub[:, 1], color=BLUE, width=0.7)
for i, c in enumerate(ub[:, 1]):
    ax.text(i, c + 2, str(int(c)), ha="center", fontsize=8.6)
ax.set_xticks(range(len(ub))); ax.set_xticklabels(lab, rotation=60, fontsize=8.5)
ax.set_ylabel("처음 신호가 잡힌 기업 수 (곳)"); ax.set_ylim(0, 135); ax.grid(axis="x", visible=False)
ax.set_title("기준월별 신호 기업 수")
finish(fig, "집계 결과 · 노트북 04·05 (v2_counts, units_by_month) · 첫 달(24.06)이 많은 것은 판정 시작 시점에 이미 진행 중이던 감소가 함께 잡히기 때문(해석 가설)")

- 신호 기업의 **월별 위험집합**(그 달에 판정 가능하지만 아직 신호가 없던 기업)은 매달 2,000~2,250곳이고, 신호 기업은 월평균 약 38곳임(전체 487곳 ÷ 13개월).
- **검정력**: 신호 기업 488곳, 비교 기업 2,023곳이면 연관의 **최소 검출 비율비(MDE)** 가 1.81임 — 즉 RR이 1.8 이상이면 통계적으로 구별할 수 있는 표본 크기임(실제 RR은 4.7).

### 4장의 결론 — 신호 규칙

| 항목 | 값 | 정한 방법 |
|---|---|---|
| 지수 | 국내 5채널 근사 건수 합 I1 | 채널 구조 점검(3장) |
| 비교 기준 / 창 길이 | 직전 기간 / **6개월** | 계절성 없음, 일시적 하락 비율이 가장 낮은 창 |
| 활동량 하한 Φ | **11.17** (월평균 근사 건수) | 변화율 IQR이 안정되는 활동량(상한 구간 칸 제외) |
| 감소 기준 n | **35%** | 변화율 하위 5% 지점(−34.6%)을 반올림 |
| 판정 기간 | 기준월 2024.06~2025.06 (13개월) | 신호 후 6개월 추적 필요 |
| 신호 | 판정 가능 & 변화율 −35% 이하로 **처음** 내려간 달 (기업당 1회) | |
| 판정 가능 기업 | 36개월 법인 3,372곳 중 **2,511곳(74.5%)** | |

---
# 5장. 통계 기법 맛보기 — 합성 예시로 이해하기(함정 절대 쓰지 말기)

6장부터는 실제 결과를 읽음. 그 전에, 결과를 읽는 데 꼭 필요한 **세 가지 통계 기법**을 **컴퓨터로 만든 가짜 자료(합성 예시)** 로 먼저 익혀 둠. 가짜 자료는 '정답'을 우리가 알고 있으므로 기법이 제대로 작동하는지 눈으로 확인할 수 있음. **이 장의 숫자는 실제 결과가 아님.**

| 절 | 기법 | 답하는 질문 |
|---|---|---|
| 5.1 | 층화(stratification)와 직접 표준화(direct standardization) | 신호 기업과 비교 기업의 **구성이 다른데** 어떻게 공정하게 비교하나? |
| 5.2 | 군집 부트스트랩(cluster bootstrap)과 95% 신뢰구간 | 얻은 RR이 **우연히 흔들릴 수 있는 범위**는? |
| 5.3 | 평균 회귀(regression to the mean)와 기준 기간 공유 | 신호와 약화가 **같은 기준 기간 때문에** 함께 움직이는 것은 아닌가? |

---
### 5.1 층화와 직접 표준화 — 구성이 다른 집단을 공정하게 비교하기

**왜 필요한가**: 신호가 잡힌 기업은 **규모가 작은 기업에 몰려** 있음(실제 자료에서 신호 기업의 46.6%가 규모 1분위, 비교 기업은 16.0%). 규모가 작은 기업은 평소에도 요구불입금액이 크게 출렁여 '약화'가 더 잦음. 그대로 비교하면 **신호 때문이 아니라 규모 때문에** 차이가 커 보일 수 있음. 이를 **교란(confounding)** 이라고 함.

**해결**
- **층화**: 비교를 같은 조건끼리만 함. 이 분석의 층은 **기준월 × 규모 5분위 × 업종군**임.
- **직접 표준화**: 층마다 구한 결과를 **신호 기업의 구성 비율**에 맞춰 합침. 즉 '**신호 기업과 똑같은 구성의 비교 기업**이었다면 약화가 몇 건이었을까(기대 건수)'를 계산함.
- **상대위험도(RR)** = **관찰된** 약화 건수 ÷ **기대** 약화 건수. RR이 1이면 연관이 없고, 4.7이면 비교 기업보다 4.7배 많이 약화됨.
- **공통 지지(common support)**: 어떤 층에 비교 기업이 5곳 미만이면 비교가 불안정하므로 업종군을 합치고, 그래도 부족하면 그 층의 신호 기업은 비교에서 뺌.

아래 합성 예시는 **'모든 규모에서 신호의 진짜 효과는 정확히 2배'**로 만들어 두었음. 신호 기업이 작은 규모에 몰려 있을 때, 그냥 비교한 값(crude)과 표준화한 값이 어떻게 달라지는지 보세요.

In [ ]:
# ---- 합성 예시: 진짜 효과(RR)는 모든 층에서 2배로 설정. 실제 자료가 아님 ----
rng = np.random.default_rng(2026)
strata_names = ["작은 규모", "중간 규모", "큰 규모"]
p_ctrl_true = np.array([0.12, 0.05, 0.02])            # 층별 비교 기업의 약화 확률 (작은 기업일수록 높음)
TRUE_RR = 2.0
n_ctrl = np.array([1600, 2000, 2400])                  # 비교 기업 수
n_sig = np.array([400, 200, 60])                       # 신호 기업 수: 작은 규모에 몰려 있음

stratum = np.repeat(np.arange(3), n_ctrl + n_sig)
is_sig = np.concatenate([np.r_[np.zeros(n_ctrl[s], bool), np.ones(n_sig[s], bool)] for s in range(3)])
p = p_ctrl_true[stratum] * np.where(is_sig, TRUE_RR, 1.0)
y = rng.random(len(p)) < p                              # 약화 여부

def crude_and_standardized(stratum, is_sig, y):
    """crude RR = 신호 전체 약화율 / 비교 전체 약화율, 표준화 RR = 관찰 약화 건수 / 기대 약화 건수"""
    S = np.unique(stratum)
    n1 = np.array([(is_sig & (stratum == s)).sum() for s in S])
    p0 = np.array([y[(~is_sig) & (stratum == s)].mean() for s in S])
    obs, exp = y[is_sig].sum(), (n1 * p0).sum()
    return y[is_sig].mean() / y[~is_sig].mean(), obs / exp, obs, exp

crude, std, obs, exp = crude_and_standardized(stratum, is_sig, y)
print(f"신호 기업 {is_sig.sum()}곳 중 약화 {obs}건 / 같은 구성의 비교 기업이었다면 기대 {exp:.1f}건")
print(f"단순 비교(crude) RR = {crude:.2f}   표준화 RR = {std:.2f}   (설정한 진짜 값 = {TRUE_RR})")

fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.3))
ax = axes[0]
sh_sig = n_sig / n_sig.sum() * 100; sh_ctl = n_ctrl / n_ctrl.sum() * 100
x = np.arange(3); w = 0.38
ax.bar(x - w / 2, sh_sig, w, color=BLUE, label="신호 기업"); ax.bar(x + w / 2, sh_ctl, w, color=GRAY, label="비교 기업")
for i in range(3):
    ax.text(i - w / 2, sh_sig[i] + 1, f"{sh_sig[i]:.0f}%", ha="center", fontsize=9.5); ax.text(i + w / 2, sh_ctl[i] + 1, f"{sh_ctl[i]:.0f}%", ha="center", fontsize=9.5)
ax.set_xticks(x); ax.set_xticklabels(strata_names); ax.set_ylim(0, 75); ax.set_ylabel("집단 안 구성 비율(%)"); ax.legend(); ax.grid(axis="x", visible=False)
ax.set_title("① 구성이 다르다: 신호 기업은 작은 규모에 몰림")

ax = axes[1]
r_sig = [y[is_sig & (stratum == s)].mean() * 100 for s in range(3)]; r_ctl = [y[(~is_sig) & (stratum == s)].mean() * 100 for s in range(3)]
ax.bar(x - w / 2, r_sig, w, color=BLUE); ax.bar(x + w / 2, r_ctl, w, color=GRAY)
for i in range(3):
    ax.text(i - w / 2, r_sig[i] + 0.6, f"{r_sig[i]:.1f}%", ha="center", fontsize=9.3); ax.text(i + w / 2, r_ctl[i] + 0.6, f"{r_ctl[i]:.1f}%", ha="center", fontsize=9.3)
    ax.text(i, max(r_sig[i], r_ctl[i]) + 3.2, f"{r_sig[i] / r_ctl[i]:.1f}배", ha="center", fontsize=10, color=ORANGE, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(strata_names); ax.set_ylim(0, 34); ax.set_ylabel("약화 비율(%)"); ax.grid(axis="x", visible=False)
ax.set_title("② 같은 층 안에서는 어디서나 약 2배")

ax = axes[2]
vals = [TRUE_RR, crude, std]; cols = [LIGHT, ORANGE, BLUE]; labs = ["진짜 값\n(설정)", "단순 비교\n(crude)", "층화·표준화"]
ax.bar(range(3), vals, color=cols, width=0.6)
for i, v in enumerate(vals):
    ax.text(i, v + 0.1, f"{v:.2f}", ha="center", fontsize=11, fontweight="bold")
ax.set_xticks(range(3)); ax.set_xticklabels(labs); ax.set_ylim(0, 4.2); ax.set_ylabel("상대위험도(RR)"); ax.grid(axis="x", visible=False)
ax.axhline(1, color=GRAY, ls="--", lw=1)
ax.set_title("③ 단순 비교는 효과를 부풀린다")
finish(fig, "합성 예시 · 가짜 자료(진짜 RR을 2로 설정). 실제 분석에서는 층이 '기준월 × 규모 5분위 × 업종군'이며 같은 방식으로 계산함")

**읽는 법**: ①에서 신호 기업은 작은 규모에 몰려 있고, ②에서 **같은 규모끼리 비교하면 어디서나 약 2배**임. 그런데 ③에서 구성을 무시하고 전체끼리 비교하면(단순 비교) **약 3배로 부풀려짐** — 신호 기업이 원래 약화가 잦은 작은 규모에 많기 때문임. **층화·표준화**를 하면 진짜 값(2배)에 가깝게 돌아옴.

> **쉽게 말해** 달리기 시합에서 한 팀은 초등학생이 대부분이고 다른 팀은 성인이 대부분이면, 전체 평균 기록만 비교해서는 안 됨. 같은 나이대끼리 비교한 뒤, 한 팀의 나이 구성에 맞춰 합치는 것이 직접 표준화임.

---
### 5.2 군집 부트스트랩과 95% 신뢰구간 — 결과가 흔들릴 수 있는 범위

**왜 필요한가**: 우리가 가진 기업은 전체 중 **일부(표본)** 임. 다른 기업들로 같은 분석을 했다면 RR이 정확히 똑같이 나오지는 않음. 이 **흔들림의 범위**를 알아야 '4.7'이라는 숫자를 얼마나 믿을 수 있는지 알 수 있음.

**부트스트랩(bootstrap)** 의 아이디어
1. 가진 표본에서 **복원추출**(뽑은 것을 다시 넣고 뽑기)로 같은 크기의 가짜 표본을 만듦.
2. 그 가짜 표본으로 RR을 다시 계산함.
3. 이를 **2,000번** 반복해 RR 2,000개를 얻음.
4. 2,000개를 줄 세운 뒤 **2.5번째와 97.5번째 백분위수**를 **95% 신뢰구간**으로 씀.

**군집(cluster)** 이란 무엇인가: 같은 기업이 **여러 기준월에 비교 기업으로 반복해서 등장**하므로 관측이 서로 독립이 아님(어떤 기업은 계속 약화되기 쉬움). 그래서 **행이 아니라 기업 단위로** 복원추출하고, 뽑힌 기업의 **모든 기준월 기록을 함께** 가져옴. 그렇지 않으면 신뢰구간이 **실제보다 좁게(과신)** 나옴.

아래 합성 예시는 '기업마다 약화되기 쉬운 정도가 크게 다르고 기업이 여러 달 반복 등장하는' 자료를 만들고, 행 단위 부트스트랩과 기업 단위(군집) 부트스트랩의 신뢰구간을 비교함. (원리를 보이기 위해 **두 집단 모두** 반복 등장하도록 만들었음. 실제 분석에서는 신호 기업은 최초 신호 1건이고 비교 기업이 여러 기준월에 반복 등장함.)

In [ ]:
# ---- 합성 예시: 기업마다 약화되기 쉬운 정도(frailty)가 크게 다르고, 기업이 여러 달 반복 등장하는 가짜 자료 ----
rng = np.random.default_rng(7)
F_SIG, F_CTL = 100, 400
frail = rng.gamma(shape=0.2, scale=1 / 0.2, size=F_SIG + F_CTL)             # 기업별 약화 경향 (평균 1, 기업 간 차이가 큼)
firm, sig, y = [], [], []
for f in range(F_SIG + F_CTL):
    is_sig = f < F_SIG                                                      # 앞 100곳 = 집단 A(신호), 나머지 = 집단 B(비교)
    for _ in range(rng.integers(1, 13)):                                    # 한 기업이 1~12개월 반복 등장
        firm.append(f); sig.append(is_sig); y.append(rng.random() < min(0.06 * (2.5 if is_sig else 1.0) * frail[f], 0.97))
firm, sig, y = np.array(firm), np.array(sig), np.array(y, dtype=float)

def rr_weighted(w):
    """가중치 w(= 뽑힌 횟수)를 준 표본에서 RR = 신호 약화율 / 비교 약화율"""
    n1, n0 = (w * sig).sum(), (w * ~sig).sum()
    return ((w * sig * y).sum() / n1) / ((w * ~sig * y).sum() / n0)

rr_hat = rr_weighted(np.ones(len(y)))
B = 2000
rr_row = np.empty(B); rr_cluster = np.empty(B); F = F_SIG + F_CTL
for b in range(B):
    w_row = np.bincount(rng.integers(0, len(y), len(y)), minlength=len(y)).astype(float)      # 행 단위 복원추출
    w_cl = np.bincount(rng.integers(0, F, F), minlength=F).astype(float)[firm]               # 기업 단위 복원추출(기업의 모든 행을 함께)
    rr_row[b], rr_cluster[b] = rr_weighted(w_row), rr_weighted(w_cl)
ci_row, ci_cl = np.percentile(rr_row, [2.5, 97.5]), np.percentile(rr_cluster, [2.5, 97.5])
print(f"기업 {F_SIG + F_CTL}곳, 기업-월 {len(y):,}행 / 관측된 RR = {rr_hat:.2f}  (설정한 진짜 값 = 2.5)")
print(f"행 단위 부트스트랩    95% 신뢰구간 {ci_row[0]:.2f} ~ {ci_row[1]:.2f}  (폭 {ci_row[1] - ci_row[0]:.2f})")
print(f"기업 단위(군집) 부트스트랩 95% 신뢰구간 {ci_cl[0]:.2f} ~ {ci_cl[1]:.2f}  (폭 {ci_cl[1] - ci_cl[0]:.2f})  ← 더 넓고 정직함")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.1), sharex=True, sharey=True)
bins = np.linspace(min(rr_row.min(), rr_cluster.min()), max(rr_row.max(), rr_cluster.max()), 60)
ymax = max(np.histogram(rr_row, bins)[0].max(), np.histogram(rr_cluster, bins)[0].max()) * 1.32
for ax, arr, ci, title, col in ((axes[0], rr_row, ci_row, "행 단위로 뽑으면 (잘못된 방법)", GRAY), (axes[1], rr_cluster, ci_cl, "기업 단위로 뽑으면 (군집 부트스트랩)", BLUE)):
    ax.hist(arr, bins=bins, color=col, alpha=0.9); ax.set_ylim(0, ymax)
    ax.axvline(ci[0], color=ORANGE, lw=1.6); ax.axvline(ci[1], color=ORANGE, lw=1.6); ax.axvline(rr_hat, color=INK, lw=1.2, ls="--")
    ax.axvspan(ci[0], ci[1], color=ORANGE, alpha=0.12)
    ax.text((ci[0] + ci[1]) / 2, ymax * 0.97, f"95% 신뢰구간\n{ci[0]:.2f} ~ {ci[1]:.2f}  (폭 {ci[1] - ci[0]:.2f})", ha="center", va="top", color=ORANGE, fontsize=10, fontweight="bold", bbox=dict(fc="white", ec="none", alpha=0.85, pad=2))
    ax.set_title(title); ax.set_xlabel("부트스트랩으로 다시 구한 RR (2,000개)"); ax.set_yticks([])
axes[0].set_ylabel("빈도")
finish(fig, "합성 예시 · 가짜 자료(집단 A 100곳·집단 B 400곳, 기업당 1~12개월 반복 등장). 점선 = 관측된 RR, 주황 = 2.5~97.5 백분위수")

**읽는 법**: 같은 자료인데도 **기업 단위(군집) 부트스트랩**의 신뢰구간이 약 2배 넓음. 기업이 여러 달 반복 등장하는 자료를 행 단위로 뽑으면 '독립적인 정보가 실제보다 많다'고 착각해서 신뢰구간이 과하게 좁아지고, 연관이 실제보다 분명해 보임(행 단위 구간의 하한은 1.6인데 기업 단위 구간의 하한은 1.1로 1에 가까움). 이 분석은 **기업 단위**로 뽑았음.

> **통계 용어 — 95% 신뢰구간을 읽는 법**
> - 점추정(RR 4.69)은 이번 표본에서 계산한 값이고, **신뢰구간(3.75~5.95)** 은 표본이 달라질 때 이 값이 움직일 수 있는 범위임.
> - **하한이 1보다 크면** '연관 없음(RR = 1)'과 구별됨 → 이 분석의 **채택 기준**.
> - 구간이 **넓을수록** 불확실함(표본이 작을 때).
> - '95%'는 같은 방법을 여러 번 반복했을 때 구간이 진짜 값을 포함하는 경우가 약 95%라는 뜻임. '진짜 값이 이 구간 안에 있을 확률이 95%'라고 말하면 엄밀하게는 틀린 표현임.

---
### 5.3 평균 회귀와 '기준 기간 공유' — 둘이 함께 움직이는 건 착시일 수 있나?

**평균 회귀(regression to the mean)**: 평소보다 **일시적으로 높았던 값**은 시간이 지나면 평소 수준으로 **되돌아오는** 경향이 있음(반대로 일시적으로 낮았던 값은 오름). 원인이 없어도 나타나는 **통계적 현상**임.

**이 분석에서 왜 문제가 되나**: 신호의 비교 기간(직전 6개월)과 약화의 기준 기간(전년 동기)이 **같은 달**임. 만약 그 기간에 어떤 기업의 **건수와 요구불입금액이 함께 일시적으로 높았다면**(예: 큰 수주), 그 뒤 평소로 돌아오는 것만으로 **둘 다 '감소'로 측정**되어 신호와 약화가 함께 나타남. 건수 감소가 입금 감소를 일으키지 않았는데도 연관이 생김.

아래 합성 예시는 **건수와 요구불입금액 사이에 아무 인과가 없도록** 만든 자료임. 다만 기준 기간에 두 값이 함께 일시적으로 높아지는 기업이 있게 했음.

In [ ]:
# ---- 합성 예시: 건수와 요구불입금액 사이에 인과 없음. 기준 기간의 일시적 호조만 공유 ----
rng = np.random.default_rng(99)
N = 20000
shock = rng.normal(0, 0.15, N)                       # 기준 기간의 '일시적 호조'(건수·금액에 함께 영향)

def log_noise(s=0.20):
    return rng.normal(0, s, N)

# (A) 기준 기간 공유: 신호의 비교 기간 = 약화의 기준 기간 (같은 shock 를 둘 다 포함)
cnt_base_A, dep_base = np.exp(shock + log_noise()), np.exp(shock + log_noise())
cnt_recent, dep_post = np.exp(log_noise()), np.exp(log_noise())                 # 이후에는 평소 수준(호조 없음)
r_A = cnt_recent / cnt_base_A - 1                                              # 건수 변화율
g = dep_post / dep_base - 1                                                    # 요구불입금액 변화율 (전년 동기 대비)

# (B) 기준 기간 비공유: 신호의 비교 기간을 다른 시기(전년 같은 달)로 → 그 시기에는 호조가 없음
cnt_base_B = np.exp(log_noise())
r_B = cnt_recent / cnt_base_B - 1

def rr_from(r, g, q=5):
    sig = r <= np.percentile(r, q); weak = g <= np.percentile(g, q)
    return weak[sig].mean() / weak[~sig].mean(), weak[sig].mean() * 100, weak[~sig].mean() * 100

rrA, pA1, pA0 = rr_from(r_A, g); rrB, pB1, pB0 = rr_from(r_B, g)
print(f"(A) 기준 기간 공유   : 신호 기업의 약화 {pA1:.1f}% vs 비신호 {pA0:.1f}%  →  RR = {rrA:.1f}   (인과가 없는데도 연관이 생김)")
print(f"(B) 기준 기간 비공유 : 신호 기업의 약화 {pB1:.1f}% vs 비신호 {pB0:.1f}%  →  RR = {rrB:.1f}   (연관이 사라짐)")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.5), sharex=True, sharey=True)
idx = rng.choice(N, 2500, replace=False)
for ax, r, name, rr, p1, p0 in ((axes[0], r_A, "(A) 기준 기간 공유", rrA, pA1, pA0), (axes[1], r_B, "(B) 기준 기간 비공유", rrB, pB1, pB0)):
    ax.scatter(r[idx] * 100, g[idx] * 100, s=7, color=GRAY, alpha=0.45, lw=0)
    sig = r <= np.percentile(r, 5); weak = g <= np.percentile(g, 5)
    ax.axvline(np.percentile(r, 5) * 100, color=BLUE, ls="--", lw=1.2); ax.axhline(np.percentile(g, 5) * 100, color=ORANGE, ls="--", lw=1.2)
    both = np.flatnonzero(sig & weak)                              # 신호이면서 약화인 기업은 전부 표시
    ax.scatter(r[both] * 100, g[both] * 100, s=14, color=ORANGE, alpha=0.9, lw=0)
    ax.set_title(f"{name}: 신호 기업의 약화 {p1:.1f}% vs 비신호 {p0:.1f}%  →  RR {rr:.1f}", fontsize=11); ax.set_xlabel("건수 변화율 r (%)  [왼쪽 점선보다 왼쪽 = 신호]")
axes[0].set_ylabel("요구불입금액 변화율 g (%)  [아래 점선보다 아래 = 약화]")
axes[0].set_xlim(-90, 120); axes[0].set_ylim(-90, 120)
finish(fig, "합성 예시 · 가짜 자료(건수와 금액 사이 인과 없음, 기준 기간에 일시적 호조만 공유). 주황 점 = 신호이면서 약화인 기업")

**읽는 법**: (A)에서는 건수와 요구불입금액이 **서로 아무 영향도 주지 않는데도** 신호와 약화가 함께 나타남(주황 점이 왼쪽 아래에 몰림). 기준 기간의 일시적 호조가 두 값을 **함께 끌어내렸기** 때문임. (B)처럼 신호의 비교 기간을 약화의 기준 기간과 **겹치지 않게** 옮기면 연관이 사라짐.

→ 그래서 실제 분석에서 이를 **직접 점검**했음. 신호를 '전년 같은 달 대비 감소'로 다시 정의해 비교 기간이 겹치지 않게 했을 때(9장), **연관이 사라지지 않고 RR 4.36으로 유지**되었음. 이는 기준 기간 공유만으로 연관이 만들어진 것은 아니라는 증거임. 다만 같은 요구불입금액 자료를 쓰므로 평균 회귀·기저 효과의 **모든 경로를 배제했다고는 말하지 않음**.

### 5장 정리

| 기법 | 한 줄 요약 | 실제 분석에서 |
|---|---|---|
| 층화·직접 표준화 | 같은 조건끼리 비교하고, 신호 기업의 구성에 맞춰 합침 | 기준월 × 규모 5분위 × 업종군, RR = 관찰 ÷ 기대 |
| 군집 부트스트랩 | 기업 단위로 다시 뽑아 RR을 2,000번 구함 | 95% 신뢰구간, 하한 > 1이면 채택 |
| 평균 회귀·기준 기간 공유 | 일시적으로 높았던 값이 평소로 돌아오는 현상이 연관처럼 보일 수 있음 | 경로 분석·급등 기업 분리·기준 기간 비공유 신호로 점검 |

---
# 6장. 핵심 결과 — 신호와 요구불입금액 약화의 연관 (노트북 05)

### 6.1 무엇을 비교했나

- **신호 기업**: 4장의 규칙으로 처음 신호가 잡힌 기업(P36 기준 **487곳** = 최초 신호 488곳 중 요구불입금액 변화율을 계산할 수 있는 기업). 기준월 = 신호가 처음 잡힌 달.
- **약화**: 신호 후 6개월 평균 요구불입금액이 **전년 같은 달 평균 대비 −64.2% 이하**(판정 가능한 모든 기업-월의 변화율 중 **하위 5%**)로 줄어든 경우.
- **비교 기업(위험집합)**: 같은 기준월에 판정 가능하지만 **그 달까지 신호가 없던 기업**(기업-월 27,621건). 신호 기업과 **같은 기준월·규모(5분위)·업종군**의 비교 기업끼리 비교함(층화), 신호 기업의 구성에 맞춰 합침(직접 표준화).
- 질문(가설 H-C1): **신호 기업의 약화 비율이 비교 기업보다 높은가?** 채택 기준은 **RR의 95% 신뢰구간 하한 > 1**임(사전 확정).

> **쉽게 말해** 건수가 줄어든 기업 487곳을 뽑고, '같은 때·같은 규모·같은 업종인데 건수는 줄지 않은 기업'이 같은 기간에 요구불입금액이 크게 줄어든 비율과 비교함. 신호 기업에서 약화가 훨씬 흔하다면 두 신호는 함께 나타난다고 할 수 있음.


(이쁘게 씀)

In [ ]:
m = MAIN
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2), gridspec_kw={"width_ratios": [1, 1.2]})
ax = axes[0]
ax.bar([0, 1], [m["p_det"], m["p_ctrl"]], color=[BLUE, GRAY], width=0.55)
ax.text(0, m["p_det"] + 0.8, f"{m['p_det']}%\n({m['obs']}곳 / {m['n1']}곳)", ha="center", fontsize=11, fontweight="bold")
ax.text(1, m["p_ctrl"] + 0.8, f"{m['p_ctrl']}%\n(기대 {m['exp']}건 / {m['n1']}곳)", ha="center", fontsize=11, fontweight="bold")
ax.set_xticks([0, 1]); ax.set_xticklabels(["신호 기업", "비교 기업\n(층화·표준화)"], fontsize=10.5)
ax.set_ylim(0, 33); ax.set_ylabel("요구불입금액 '약화' 비율(%)"); ax.grid(axis="x", visible=False)
ax.annotate("", xy=(0.82, m["p_ctrl"] + 5), xytext=(0.82, m["p_det"] - 1), arrowprops=dict(arrowstyle="<->", color=ORANGE, lw=1.5))
ax.text(1.0, 16, f"약 {m['rr']:.1f}배", color=ORANGE, fontsize=13, fontweight="bold", ha="left", va="center")
ax.set_title("신호 기업에서 약화가 훨씬 흔했다")

ax = axes[1]
forest(ax, [("신호 기업 vs\n비교 기업", m["rr"], m["lo"], m["hi"], BLUE)], xmax=8, ref=1.0)
ax.axvspan(0, 1, color=PALE_ORANGE, alpha=0.35, zorder=0)
ax.text(0.5, 0.62, "1 이하 =\n연관 없음/반대", ha="center", va="center", fontsize=9, color=ORANGE)
ax.set_ylim(-1.6, 1.6); ax.set_yticks([0]); ax.set_yticklabels(["신호 기업 vs\n비교 기업"])
ax.annotate("RR 4.69\n95% 신뢰구간 3.75 ~ 5.95", xy=(m["rr"], 0.12), xytext=(m["rr"] - 1.6, 0.95), fontsize=10.5, color=BLUE, fontweight="bold", ha="center", arrowprops=dict(arrowstyle="-", color=BLUE))
ax.text(8, -1.15, f"하한 {m['lo']:.2f} > 1 → 연관 없음과 구별됨 (채택)\n부트스트랩 {m['B']:,}회 중 RR ≤ 1이 나온 적 없음 (단측 p = {m['p_one']})", ha="right", fontsize=9.5, color=INK)
ax.set_title("상대위험도(RR)와 95% 신뢰구간")
finish(fig, "집계 결과 · 노트북 05 (hc1_main) · 표시된 24.6%와 5.3%는 반올림값이라 단순 나눗셈(4.64)과 RR 4.69가 약간 다름(RR은 반올림 전 값으로 계산)")

**읽는 법과 결과**

| 항목 | 값 |
|---|---|
| 신호 기업 중 약화 | **120곳 / 487곳 = 24.6%** (관찰 건수) |
| 같은 구성의 비교 기업이었다면 | 기대 25.6건 (= 5.3%) |
| 상대위험도 RR | **4.69**, 95% 신뢰구간 **3.75 ~ 5.95** |
| 판정 | 신뢰구간 하한 > 1 → **가설 채택** (신호와 약화는 함께 나타남) |

- **상대위험도(relative risk, RR)** = 관찰된 약화 건수 ÷ 기대 약화 건수. 1이면 연관 없음, 4.69이면 신호 기업의 약화가 같은 구성의 비교 기업보다 **약 4.7배** 많다는 뜻임.
- **절대 위험도 함께 읽기**: 신호 기업의 약화 비율은 **24.6%**임. 즉 신호 기업의 약 **75%는 약화에 해당하지 않음**. RR이 크다고 신호 기업 대부분이 약화한다는 뜻이 아님(9장에서 더 자세히 다룸).
- **연관이지 인과가 아님**: 건수가 줄어서 요구불입금액이 줄었다고 말하는 것이 아님.

### 6.2 독립 검증 표본에서도 같은가 (홀드아웃)

규칙과 기준값을 **정하는 데 쓰지 않은** 35개월 법인(P35만 1,412곳)에 **같은 규칙을 그대로** 적용했음.

(씀)

In [ ]:
rows = []
cols = [BLUE, PURPLE, GRAY]
for (lab, n1, rr, lo, hi, pd_, pc_), c in zip(SAMPLE_RR, cols):
    rows.append((f"{lab}\n신호 기업 {n1}곳 · 약화 {pd_}% vs {pc_}%", rr, lo, hi, c))
fig, ax = plt.subplots(figsize=(11, 3.9))
forest(ax, rows, xmax=8, base=MAIN["rr"])
ax.set_title("규칙을 정하지 않은 독립 표본에서도 같은 방향·비슷한 크기")
fig.subplots_adjust(right=0.82)
finish(fig, "집계 결과 · 노트북 05 (holdout_main) · 주황 점선 = P36의 RR 4.69 · P35+는 P36을 포함하므로 독립 증거가 아님(표본이 늘어 신뢰구간이 가장 좁음)", rect=(0, 0.04, 0.82, 1))

- 세 RR(4.69 / 4.29 / 4.59)이 **같은 방향·비슷한 크기**임. P35만의 신호 기업은 265곳으로 적어 신뢰구간이 더 넓지만 하한이 3.28로 1보다 훨씬 큼.
- **두 RR의 차이**는 로그 RR의 차이(= 비의 로그)로 비교함 — P36과 P35만의 차이는 **−0.09 (95% 신뢰구간 −0.44 ~ +0.25)** 로 **0을 포함**하므로 두 값은 **통계적으로 구별되지 않음**. (구별되지 않는다는 것이 '같다'는 증명은 아님.)
- **한계**: P35만은 **같은 기간(2023.01~2025.12)의 다른 기업**임. 시간이 지나도 성립하는지(시간적 독립 검증)는 아님. 기준월을 나누면 RR이 2024년 4.05, 2025년 5.78로 달라져 시기별 변동이 있음(8장).

### 6.3 비교 집단이 정말 비슷해졌나 — 균형 점검

> **통계 용어 — 표준화 차이(standardized difference)**
> 두 집단의 평균(비율) 차이를 표준편차로 나눈 값임. 표본 크기에 영향을 받지 않아 **'구성이 얼마나 다른가'** 를 재는 데 쓰임. 절댓값이 **0.1 이하**이면 균형이 잡혔다고 봄.
> 층화·표준화를 했다면 **조정 후**에는 규모·업종처럼 층에 쓴 변수의 차이가 0이 되어야 하고, 층에 쓰지 않은 변수(고객등급·전담 여부·지역 등)는 얼마나 남는지 확인함.


(안씀)

In [ ]:
bal = pd.DataFrame(BALANCE, columns=["변수", "신호", "비교(조정 전)", "비교(표준화)", "차이 전", "차이 후"])
bal["전"] = bal["차이 전"].abs(); bal["후"] = bal["차이 후"].abs()
bal = bal.sort_values("전", ascending=True).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(10.5, 8.2))
ax.axvspan(0, 0.1, color="#E8F0FA", zorder=0); ax.text(0.05, len(bal) - 0.35, "균형 (0.1 이하)", ha="center", fontsize=9, color=BLUE)
for i, r in bal.iterrows():
    ax.plot([r["후"], r["전"]], [i, i], color=LIGHT, lw=2, zorder=1)
    ax.plot(r["전"], i, "o", color="white", mec=GRAY, mew=1.6, ms=8, zorder=2)
    col = ORANGE if r["후"] > 0.1 else BLUE
    ax.plot(r["후"], i, "o", color=col, ms=8, zorder=3)
    if r["후"] > 0.1:
        ax.text(r["후"] + 0.02, i, f"{r['후']:.2f}", va="center", fontsize=9, color=ORANGE)
ax.set_yticks(range(len(bal))); ax.set_yticklabels(bal["변수"], fontsize=9.3)
ax.set_xlim(0, 0.78); ax.set_xlabel("표준화 차이의 절댓값 (신호 기업 vs 비교 기업)"); ax.grid(axis="y", visible=False)
ax.plot([], [], "o", color="white", mec=GRAY, mew=1.6, ms=8, label="조정 전 (단순 비교)"); ax.plot([], [], "o", color=BLUE, ms=8, label="층화·표준화 후"); ax.plot([], [], "o", color=ORANGE, ms=8, label="조정 후에도 0.1 초과")
ax.legend(loc="lower right")
ax.set_title("층화·표준화로 규모·업종은 정확히 맞춰졌고, 일부 변수는 차이가 남는다")
finish(fig, "집계 결과 · 노트북 05 (balance) · 규모·업종군은 층에 쓴 변수라 조정 후 0, 나머지는 층에 쓰지 않아 남은 차이를 그대로 보여 줌")
left = bal[bal["후"] > 0.1][["변수", "후"]]
print("조정 후에도 표준화 차이가 0.1을 넘는 변수:", ", ".join(f"{v}({d:.2f})" for v, d in zip(left["변수"], left["후"])))

**읽는 법**: 흰 동그라미(조정 전)에서 파란 점(조정 후)으로 선이 움직임. **규모 1·4·5분위**는 조정 전 차이가 0.4~0.7로 컸지만(신호 기업이 작은 규모에 몰림) 조정 후 0이 됨. 그러나 **고객등급(최우수)·전담 고객 여부·지역·탐지 전 요구불입금액 추세**는 층에 쓰지 않았기 때문에 **0.11~0.18의 차이가 남음**(주황).

- 즉 **잔여 불균형**이 있어 이 변수들의 영향은 RR에 반영되지 않았음. 한계로 명시함(11장).
- 이 중 탐지 전 요구불입금액 추세는 **직접 조정**하는 보조 분석(S1)에서 따로 확인했음(RR 5.26, 아래 6.6).

### 6.4 특정 기준월에 좌우되나 — 기준월 하나씩 빼기 (leave-one-out)

13개 기준월 중 **한 달씩 빼고** RR을 다시 계산했음. 어느 한 달이 결과를 끌고 가는지 확인하는 방법임.

(씀)

In [ ]:
lo_ = np.array(LOMO)
fig, ax = plt.subplots(figsize=(10.5, 3.9))
labs = [f"{str(int(m))[2:4]}.{str(int(m))[4:]}" for m in lo_[:, 0]]
ax.plot(range(len(lo_)), lo_[:, 2], color=BLUE, marker="o", lw=2, ms=7)
for i, v in enumerate(lo_[:, 2]):
    ax.text(i, v + 0.07, f"{v:.2f}", ha="center", fontsize=9)
ax.axhline(MAIN["rr"], color=ORANGE, ls="--", lw=1.3); ax.text(len(lo_) - 0.6, MAIN["rr"] + 0.28, f"전체 기준월 사용: RR {MAIN['rr']}", color=ORANGE, ha="right", fontsize=9.8)
ax.set_xticks(range(len(lo_))); ax.set_xticklabels(labs); ax.set_ylim(3.6, 5.5)
ax.set_xlabel("제외한 기준월 (연.월)"); ax.set_ylabel("나머지로 다시 계산한 RR")
ax.set_title(f"어느 한 달을 빼도 RR은 {lo_[:, 2].min():.2f} ~ {lo_[:, 2].max():.2f}")
finish(fig, "집계 결과 · 노트북 05 (leave_one_month)")

**읽는 법**: 한 달을 빼도 RR이 4.5~4.9 범위에서 거의 움직이지 않음. 특정 달 하나가 결과를 만든 것이 아님.

### 6.5 신호 전후로 요구불입금액은 어떻게 움직였나 — 경로 분석 (탐색)

신호가 요구불입금액보다 **먼저** 움직이는지, 아니면 **함께** 움직이는지 보려고 신호 전·중·후의 요구불입금액 흐름을 그려 봤음. **확증이 아닌 탐색(기술)** 이며 판정 기준이 없음.

- 상대 월 `j`: 신호가 처음 잡힌 달(기준월 e)을 0으로 둠. `j = −12 ~ −6`은 **건수 감소 이전**(신호의 비교 기간), `−5 ~ 0`은 **신호 판정 창**, `1 ~ 6`은 **신호 이후**임.
- **Δ(j)** = (신호 기업 요구불입금액의 전년 같은 달 대비 비) − (표준화한 비교 기업의 같은 비). **0이면 두 집단이 같은 흐름**, **양수이면 신호 기업의 요구불입금액이 (전년 대비) 더 높음**.

(이쁘게 시도)

In [ ]:
pt = pd.DataFrame([(j, g, d, c[0], c[1]) for j, g, d, c in PATH], columns=["j", "구간", "d", "lo", "hi"])
YMIN, YMAX = -1.5, 6.0
fig, ax = plt.subplots(figsize=(11.5, 4.7))
ax.axvspan(-12.5, -5.5, color=LIGHT, alpha=0.55, zorder=0); ax.axvspan(-5.5, 0.5, color=PALE_BLUE, alpha=0.55, zorder=0); ax.axvspan(0.5, 6.5, color=PALE_ORANGE, alpha=0.55, zorder=0)
ax.text(-9, YMAX - 0.25, "건수 감소 이전\n(비교 기준 기간)", ha="center", va="top", fontsize=10, color=MUTED, fontweight="bold")
ax.text(-2.5, YMAX - 0.25, "신호 판정 창\n(건수가 줄어드는 중)", ha="center", va="top", fontsize=10, color=BLUE, fontweight="bold")
ax.text(3.5, YMAX - 0.25, "신호 이후 6개월", ha="center", va="top", fontsize=10, color=ORANGE, fontweight="bold")
ax.axhline(0, color=INK, lw=1)
for _, r in pt.iterrows():
    lo, hi = max(r["lo"], YMIN), min(r["hi"], YMAX)
    ax.plot([r["j"], r["j"]], [lo, hi], color=GRAY, lw=1.6, zorder=2)
    if r["hi"] > YMAX:
        ax.plot([r["j"]], [YMAX], marker="^", color=GRAY, ms=5, zorder=2)
    if r["lo"] < YMIN:
        ax.plot([r["j"]], [YMIN], marker="v", color=GRAY, ms=5, zorder=2)
ax.plot(pt["j"], pt["d"], color=BLUE, lw=1.6, zorder=3); ax.plot(pt["j"], pt["d"], "o", color=BLUE, ms=7, mec="white", zorder=4)
ax.set_xticks(range(-12, 7)); ax.set_xlim(-12.6, 6.6); ax.set_ylim(YMIN, YMAX)
ax.set_xlabel("신호가 처음 잡힌 달(e)을 0으로 한 상대 월 j"); ax.set_ylabel("Δ(j) = 신호 기업 - 비교 기업\n(요구불입금액의 전년 같은 달 대비 비)")
ax.set_title("신호 이전에는 신호 기업의 요구불입금액이 오히려 높았고, 이후에는 차이가 사라진다")
finish(fig, "집계 결과 · 노트북 05 (hc2_path) · 회색 선 = 95% 신뢰구간(범위를 넘으면 ▲▼로 잘림) · 극단값을 제한(윈저화)한 합의 비로 계산 · 탐색 결과이며 확증 판정 없음")
for name, est, ci in PATH_SUMMARY:
    print(f"{name}: {est:.2f}  (95% 신뢰구간 {ci[0]:.2f} ~ {ci[1]:.2f})")

**읽는 법과 해석**

- **건수 감소 이전(j −12~−6)**: Δ의 평균이 **+2.02**(95% 신뢰구간 1.25 ~ 36.7, 0보다 큼) — 신호 기업의 요구불입금액이 비교 기업보다 **오히려 높았음**.
- **신호 이후(j 1~6)**: Δ가 **0 근처**(평균 +0.17, 신뢰구간이 0을 포함)로, 이전의 '높음'이 사라졌음. (신뢰구간이 매우 넓어 개별 달의 값은 불안정함.)
- **해석(가설)**: 신호가 잡히기 전에 요구불입금액이 일시적으로 높았다가 평소로 돌아오는 **평균 회귀**의 모습일 수 있음. 그래서 이 연관을 **'선행 신호(조기경보)'로 해석하지 않고**, 건수와 요구불입금액이 **함께 나타나는 신호**라고 표현함.

### 6.6 보조·사후 분석으로 해석을 점검하기

주 결과(RR 4.69)가 다른 정의와 하위 집단에서도 비슷한지, 반대 방향 변동은 어떤지 확인했음. (**사후 탐색**은 결과를 본 뒤 추가한 것이라 판정은 바꾸지 않고 기술만 함.)

| 분석 | 확인하려는 것 |
|---|---|
| 잔액 약화로 정의 | '요구불입금액'이 아닌 **예금잔액**이 하위 5%로 줄어든 경우도 신호와 연결되는가 |
| 입금 급증 꼬리(반대 방향) | 신호 기업은 요구불입금액이 크게 **늘어난** 경우가 오히려 드문가 — 단순한 '양방향 변동성'이 아닌지 |
| 2024.12 이후 기준월 | 탐지 전 요구불입금액 추세를 계산할 수 있는 부분집합(신호 기업 193곳) |
| 요구불입금액 수준·추세 직접 조정(S1) | 그 추세 차이를 보정해도 RR이 유지되는가 |
| 직전 급등 여부로 분리 | 신호 직전에 요구불입금액이 **급등**한 기업(평균 회귀 후보)을 빼도/따로 봐도 연관이 유지되는가 |

(안씀)

In [ ]:
sg = {r[0]: r for r in SURGE}
rows = [
    ("주 분석 (입금 약화 하위 5%)", MAIN["rr"], MAIN["lo"], MAIN["hi"], BLUE),
    ("잔액 약화로 정의 (보조 결과)", RR_O3[0], RR_O3[1], RR_O3[2], BLUE),
    ("입금 급증 꼬리: 크게 '늘어난' 경우 (반대 방향)", UPDOWN[1][4][0], UPDOWN[1][4][1], UPDOWN[1][4][2], ORANGE),
    ("2024.12 이후 기준월만 (신호 기업 193곳)", sg["전체 (e ≥ 23)"][4][0], sg["전체 (e ≥ 23)"][4][1], sg["전체 (e ≥ 23)"][4][2], PURPLE),
    ("  └ 요구불입금액 수준·추세 직접 조정 (S1)", S1[2][2][0], S1[2][2][1], S1[2][2][2], PURPLE),
    ("  └ 직전 급등 없는 기업 (140곳)", SURGE[1][4][0], SURGE[1][4][1], SURGE[1][4][2], PURPLE),
    ("  └ 직전 급등 기업 (53곳)", SURGE[2][4][0], SURGE[2][4][1], SURGE[2][4][2], PURPLE),
]
fig, ax = plt.subplots(figsize=(11.8, 4.6))
forest(ax, rows, xmax=10, ref=1.0, base=MAIN["rr"], label_fs=9.8)
fig.subplots_adjust(left=0.36, right=0.86)
ax.set_title("정의·기간·하위 집단을 바꿔도 같은 방향 — 반대 방향(급증)은 오히려 드물다")
finish(fig, "집계 결과 · 노트북 05 (aux_o2_o3, explore_updown, explore_surge, s1_and_normal) · 주황 점선 = 주 분석 RR 4.69 · 사후 탐색(결과를 본 뒤 추가)은 판정을 바꾸지 않음", rect=(0, 0.04, 0.86, 1))

**읽는 법과 해석**

- **잔액 약화**로 정의해도 RR 2.37(하한 > 1)로 같은 방향임.
- **입금 급증 꼬리**: 요구불입금액이 크게 **늘어난** 비율은 신호 기업 4.9%, 비교 기업 8.6%로 **신호 기업에서 오히려 드묾**(RR 0.57, 1보다 작음). 건수가 줄어든 기업이 요구불입금액이 '크게 변동'하는 것이 아니라 **하락 방향으로만** 연관이 있다는 뜻임.
- **2024.12 이후 부분집합**(193곳)에서 요구불입금액 수준·추세를 **직접 조정**해도 RR이 5.26으로 유지되고, 직전 급등 기업을 **빼도(5.39) 따로 봐도(4.16)** 연관이 남음. 직전 급등 기업의 비중은 신호 기업 27.5%, 비교 기업 9.3%로, 평균 회귀 후보가 신호 기업에 더 많이 섞여 있음은 확인됨.
- 다만 이 점검은 **결과를 본 뒤 추가한 탐색**이고 같은 요구불입금액 자료를 쓰므로 **평균 회귀·기저 효과의 기여를 완전히 배제하지는 못함**(9장에서 기준 기간을 공유하지 않는 신호로 한 번 더 점검).

### 6장 정리

- 신호 기업의 약화 비율 **24.6%**, 같은 구성의 비교 기업 **5.3%** → **RR 4.69 (3.75~5.95)**, 가설 채택.
- 규칙을 정하지 않은 **독립 표본에서 RR 4.29 (3.28~5.63)**로 재현, 두 값은 구별되지 않음.
- 특정 기준월에 좌우되지 않음(RR 4.51~4.87). 규모·업종은 맞췄으나 고객등급·전담 여부·지역 등 **잔여 불균형**이 남음.
- 신호 이전에는 요구불입금액이 오히려 높았으므로(평균 회귀 가능성) **'조기경보'가 아닌 '함께 나타나는 신호'** 로 해석함.

---
# 7장. 고객별 차이 — 어떤 고객에서 강하고 약한가 (노트북 05·06)

### 7.1 규모별 (거래 활동량 5분위)

여기서 **규모는 기업의 자산·매출이 아니라 '거래 활동량'** 임 — 신호를 판정하는 시점의 직전 6개월 월평균 근사 건수를 5등분한 값(경계 약 25·43·60·68건)임. 4~5분위는 건수 상한 근처임.

(이쁘게 시도)

In [ ]:
names = ["1분위(작음)", "2분위", "3분위", "4분위", "5분위(큼)"]
rows = [(f"{nm}  ·  신호 {x[1]}곳, 약화 {x[2]}건", x[4][0], x[4][1], x[4][2], BLUE if i < 4 else GRAY) for i, (nm, x) in enumerate(zip(names, SIZE_RR))]
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.6), gridspec_kw={"width_ratios": [1.7, 1]})
ax = axes[0]
forest(ax, rows, xmax=13, ref=1.0, base=MAIN["rr"], show_text=True, label_fs=9.8)
ax.set_title("규모별 RR — 모든 규모에서 하한이 1보다 큼 (5분위는 해석 보류)")
ax = axes[1]
det_rates = [x[3] for x in SIZE_RR]
bars = ax.bar(range(5), det_rates, color=[BLUE] * 4 + [GRAY], width=0.62)
for i, v in enumerate(det_rates):
    ax.text(i, v + 1, f"{v:.1f}%", ha="center", fontsize=10)
ax.set_xticks(range(5)); ax.set_xticklabels(["1분위", "2분위", "3분위", "4분위", "5분위"]); ax.set_ylim(0, 52); ax.grid(axis="x", visible=False)
ax.set_ylabel("신호 발생 비율(%)")
ax.set_title("규모가 클수록 신호가 드묾")
fig.subplots_adjust(wspace=0.62, left=0.17, right=0.96)
finish(fig, "집계 결과 · 노트북 05 (heterogeneity) · 5분위는 신호 기업 14곳·약화 5건뿐이라 RR 31.3의 크기는 해석하지 않음(방향만 확인) · 주황 점선 = 전체 RR 4.69")

**읽는 법과 해석**

- **방향은 동일**: 모든 규모에서 신뢰구간 하한이 1보다 큼. 크기는 3.6~6.6 사이에서 움직이며, 가장 큰 규모(5분위)는 신호 기업 14곳·약화 5건뿐이라 RR 31.3(10.5~142.9)은 **크기를 해석하지 않음**.
- **신호 발생 비율이 규모에 따라 43.4% → 3.0%로 크게 줄어듦.** 거래가 아주 많은 기업은 건수 구간의 **상한(50건 초과)에 닿아** 있어 거래가 줄어도 구간 코드가 그대로 남기 쉽기 때문임. → **거래 활동량이 큰 고객은 건수 신호만으로는 감지가 어려운 '사각지대'** 임.
- 상한 구간 채널이 있는 기업의 RR은 6.30(4.13~8.94), 없는 기업은 4.06(3.02~5.51)으로 둘 다 연관이 있음(아래 표).

(안씀)

In [ ]:
rows = [(("상한 구간 채널 있음" if "보유" in lab else "상한 구간 채널 없음") + f"\n신호 {n}곳, 약화 {ev}건", c[0], c[1], c[2], ORANGE if "보유" in lab else BLUE) for lab, n, ev, c in CEIL_RR]
fig, ax = plt.subplots(figsize=(11, 2.9))
forest(ax, rows, xmax=11, base=MAIN["rr"])
ax.set_title("상한 구간(50건 초과) 채널이 있는 기업과 없는 기업 모두 연관이 있다")
fig.subplots_adjust(right=0.8)
finish(fig, "집계 결과 · 노트북 05 (heterogeneity) · 주황 점선 = 전체 RR 4.69", rect=(0, 0.05, 0.8, 1))

### 7.2 채널 이용 유형별, 자동이체 수준별 (노트북 06)

신호와 별개로 기업을 **2023년 상반기(신호 산출 기간과 겹치지 않는 기간)의 채널 이용 습관**으로 분류했음.

| 구분 | 정의 |
|---|---|
| **채널 유형** | 오프라인(창구+ATM) 비중으로 3등분 — **인터넷·스마트 중심형** / **혼합형** / **창구·ATM 중심형** |
| **자동이체 수준** | 평균 자동이체 코드가 중앙값(5) 이상이면 **높음**, 미만이면 **낮음** |

각 집단 안에서 **같은 집단의 비신호 기업**과 비교한 RR을 구했음(같은 층화·표준화).

> **통계 용어 — RR의 비(ratio of RRs)**
> 두 집단의 RR이 다른지 보려고 **RR을 서로 나눈 값**을 씀. 비가 1이면 두 집단의 RR이 같고, 이 비의 **95% 신뢰구간이 1을 포함하면 차이가 있다고 말하기 어려움**(로그 변환해서 계산).

(안씀)

In [ ]:
order = ["인터넷·스마트 중심형", "혼합형", "창구·ATM 중심형", "자동이체 낮음", "자동이체 높음"]
g36 = {g[0]: g for g in GROUP_RR}; g35 = {g[0]: g for g in GROUP_RR_P35}
rows = []
for g in order:
    n, c = g36[g][1], g36[g][2]
    rows.append((f"{g}  [P36] 신호 {n}곳", c[0], c[1], c[2], BLUE))
    if g in g35:
        n2, c2 = g35[g][1], g35[g][2]
        rows.append((f"└ [P35만] 신호 {n2}곳", c2[0], c2[1], c2[2], PURPLE))
fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.4), gridspec_kw={"width_ratios": [1.7, 1]})
ax = axes[0]
forest(ax, rows, xmax=12, base=MAIN["rr"], label_fs=9.2)
ax.set_title("집단별 RR: 모두 연관이 있고, 독립 표본(P35만)도 같은 방향")
ax = axes[1]
rr_rows = [("창구·ATM 중심형 ÷\n인터넷·스마트 중심형", *RATIOS[1][1], BLUE), ("혼합형 ÷\n인터넷·스마트 중심형", *RATIOS[2][1], BLUE), ("자동이체 높음 ÷ 낮음", *RATIOS[3][1], BLUE)]
forest(ax, rr_rows, xmax=4.0, ref=1.0, xlabel="RR의 비 · 1 = 두 집단의 RR이 같음", label_fs=9.2)
ax.set_title("집단 간 RR의 비 (P36)")
fig.subplots_adjust(wspace=1.05, left=0.24, right=0.92)
finish(fig, "집계 결과 · 노트북 06 (groups_both, overall) · 주황 점선 = 전체 RR 4.69")

**읽는 법과 해석**

- **세 채널 유형 모두 RR이 3.5~6.1**이고 신뢰구간 하한이 1보다 큼. 독립 표본(P35만)에서도 같은 방향임 → **신호 규칙은 채널 유형과 무관하게 연관을 보임.**
- 유형 간 **RR의 비**는 1.06(0.58~1.93), 1.71(0.95~3.28)로 **모두 1을 포함** → 유형 간 크기 차이는 **확인되지 않음**. (같다고 확정할 근거는 아니며, 세 유형에 같은 신호 규칙을 적용해 볼 수 있다는 뜻임.)
- **자동이체 높음**의 RR(5.60)이 낮음(3.44)보다 큼(비 1.63, 신뢰구간 1.02~2.61로 약한 단서). 아래에서 이유를 확인함.

(안씀)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
ax = axes[0]
x = np.arange(2); w = 0.36
sig_r = [r[2] for r in AUTO_RATES]; ctl_r = [r[3] for r in AUTO_RATES]
ax.bar(x - w / 2, sig_r, w, color=BLUE, label="신호 기업"); ax.bar(x + w / 2, ctl_r, w, color=GRAY, label="비교 기업(같은 집단의 비신호, 표준화)")
for i in range(2):
    ax.text(i - w / 2, sig_r[i] + 0.6, f"{sig_r[i]:.1f}%", ha="center", fontsize=10.5); ax.text(i + w / 2, ctl_r[i] + 0.6, f"{ctl_r[i]:.1f}%", ha="center", fontsize=10.5)
    ax.text(i, max(sig_r[i], ctl_r[i]) + 4.2, f"RR {({g[0]: g[2][0] for g in GROUP_RR})[AUTO_RATES[i][0]]:.2f}", ha="center", color=ORANGE, fontsize=11, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(["자동이체 낮음", "자동이체 높음"]); ax.set_ylim(0, 40); ax.set_ylabel("약화 비율(%)"); ax.legend(loc="upper right", fontsize=9); ax.grid(axis="x", visible=False)
ax.set_title("RR이 큰 이유는 비교 기업의 약화가 드물기 때문")

ax = axes[1]
std_rate = {r[0]: r for r in TYPE_STD_RATE}
names = ["인터넷·스마트 중심형", "혼합형", "창구·ATM 중심형"]
y = np.arange(3)[::-1]
for yi, nm in zip(y, names):
    r = std_rate[nm]
    ax.plot(r[2], [yi, yi], color=BLUE, lw=2.4); ax.plot([r[1]], [yi], "o", color=BLUE, ms=9, mec="white", label="규모를 맞춘 비율" if yi == y[0] else None)
    ax.plot([r[3]], [yi - 0.22], "s", color=GRAY, ms=6, label="단순 비율(규모 안 맞춤)" if yi == y[0] else None)
    ax.text(r[2][1] + 0.5, yi, f"{r[1]:.1f}%", va="center", fontsize=10)
ax.set_yticks(y); ax.set_yticklabels(names); ax.set_xlim(10, 30); ax.set_ylim(-0.7, 2.6); ax.grid(axis="y", visible=False)
ax.set_xlabel("신호 발생 비율(%) · 점 = 추정, 선 = 95% 범위"); ax.legend(loc="lower right", fontsize=9)
ax.set_title("규모를 맞추면 유형 간 신호 발생 비율 차이가 작다")
finish(fig, "집계 결과 · 노트북 06 (c3_auto_rr, c1_standardized)")

**읽는 법과 해석**

- **왼쪽**: 자동이체가 많은 기업은 평소 요구불입금액이 안정적이라 **신호가 없는 기업의 약화 비율이 3.9%**로, 자동이체가 적은 기업(8.0%)의 절반 수준임. 이런 기업에서 신호가 잡히면 상대적으로 더 이례적이라 **RR이 더 커 보임**(5.6 vs 3.4). 그러나 **신호 기업의 실제 약화 비율은 오히려 낮음**(22.0% vs 27.6%). RR이 큰 것은 **분모(비교 기업의 약화)가 작기 때문**임 → 약한 단서로만 쓰고 **우선순위의 근거로는 쓰지 않음**.
- **오른쪽**: 규모를 맞추면 유형별 신호 발생 비율은 16~22%로 **뚜렷한 차이가 없음**. 오프라인 병행 유형은 거래가 적어 **판정 가능 비율이 낮음**(아래 그림). 즉 신호가 얼마나 잡히는지는 유형보다 **거래 활동량(규모)** 에 좌우됨.

(안씀)

In [ ]:
ctx = TYPE_CONTEXT
names = [c[0] for c in ctx]
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.6), sharey=True)
for ax, idx, title, col in ((axes[0], 2, "판정 가능 비율(%)", BLUE), (axes[1], 3, "자동이체 높음 비율(%)", PURPLE), (axes[2], 4, "외환 이용 비율(%)", ORANGE)):
    vals = [c[idx] for c in ctx]
    ax.barh(range(3), vals, color=col, height=0.55)
    for i, v in enumerate(vals):
        ax.text(v + 1.5, i, f"{v:.1f}", va="center", fontsize=10)
    ax.set_xlim(0, 100); ax.set_title(title, fontsize=11.5); ax.grid(axis="y", visible=False)
axes[0].set_yticks(range(3)); axes[0].set_yticklabels(names); axes[0].invert_yaxis()
finish(fig, "집계 결과 · 노트북 06 (type_context) · 36개월 법인 · 판정 가능 = 활동량 하한 이상인 기업")

### 7.3 채널만 옮기는 기업은 얼마나 되나 (채널 구성 이동)

**채널 이동**이란 총 거래활동량(I1)은 ±20% 이내로 유지된 채 **오프라인(창구·ATM) 비중만 0.2 이상 줄어든** 경우임. 이런 기업은 거래가 줄어든 것이 아니라 **방식이 바뀐 것**이므로 합산 신호가 오탐으로 잡지 않는지 확인했음.

- 36개월 법인 중 해당 기업은 **37곳(판정 가능 기업의 약 1.5%)** 으로 드묾. 이 중 요구불입금액이 크게 줄어든 기업은 **4곳 이하**였음(신호 기업은 24.6%).
- 표본이 너무 작아(30곳 미만 또는 약화 5건 미만) **RR은 계산하지 않고** 기술만 함. '약화가 적으니 위험이 없다'는 뜻은 아님.
- 비중 기준이 '오프라인 비중의 상위 1/3 경계(약 12%)'에서 나왔으므로, **'오프라인 중심'이 아니라 '오프라인을 상대적으로 더 쓰는 기업'** 임. 인터넷·스마트 위주 유형은 오프라인 비중이 2% 이하로 사실상 온라인 전용임.
- 결론: **채널만 옮겨서 생기는 오탐은 드물어** 합산 신호의 설계(3장)와 일치함. 이동은 신호의 오탐 원인이라기보다 **별도의 이동 신호로 분류해 원인을 확인**하는 대상으로 둠(10장).

### 7장 정리

| 고객 구분 | 결과 |
|---|---|
| 규모 | 모든 규모에서 연관, 단 규모가 클수록 **신호가 드묾**(43.4% → 3.0%) → 거래 활동량이 큰 고객은 사각지대 |
| 채널 유형 | 세 유형 모두 RR 3.5~6.1, **유형 간 차이는 확인되지 않음** |
| 자동이체 | RR은 높음이 더 크나 신호 기업의 실제 약화율은 오히려 낮음 → **약한 단서**, 우선순위 근거로 쓰지 않음 |
| 채널 이동 | 37곳(1.5%)으로 **드묾** |

---
# 8장. 분석 선택을 바꿔도 같은가 — 민감도 분석 (노트북 07)

> **통계 용어 — 민감도 분석(sensitivity analysis)**
> 분석에는 **연구자가 정한 선택**(가정·기준·정의·표본)이 많이 들어감. 그 선택을 **한 가지씩 바꿔 다시 계산**했을 때 결론이 유지되는지 보는 방법임.
> 결론이 선택마다 뒤집힌다면 '그 선택에 의존한 결과'이고, 유지된다면 **선택에 덜 민감한(견고한, robust) 결과**임.
> (여기서 '민감도'는 진단검사의 sensitivity(재현율)와 다른 뜻임.)

이 분석은 **한 번에 한 가지씩** 모두 **20가지**로 바꿔 RR과 95% 신뢰구간을 다시 구했음(여러 개를 동시에 바꾸는 조합은 하지 않음). **변형 결과를 보고 기준을 바꾸지는 않았음.**

| 묶음 | 무엇을 바꿨나 | 왜 확인하나 |
|---|---|---|
| ① 구간→건수 변환 가정 | 50건초과를 51·100건으로, 구간 하한·상한 합산, 구간 번호(순서 코드)를 그대로 합산 | 가정(60건)이 결과를 좌우하는지 |
| ② 활동량 하한 | 원래 규칙(v1), 하한을 거의 없앰 | 작은 기업을 거르는 방식이 결과를 좌우하는지 |
| ③ 약화(결과) 정의 | 하위 10%·2.5%로 넓히거나 좁힘, 추적 기간 3개월 | '약화'의 정의에 의존하는지 |
| ④ 감소 기준·비교 기간 | n = 25%·45%, 비교 기간 k = 3개월 | 신호 규칙의 기준값에 의존하는지 |
| ⑤ 비교 기업·층 | 한 번도 신호가 없던 기업만 비교군으로, 층에서 기준월을 합치거나 업종군을 뺌 | 비교 방식에 의존하는지 |
| ⑥ 표본·기간 | 35개월 법인만, 35개월 이상 전체, 2024년·2025년 기준월만 | 표본·시기에 의존하는지 |

(안씀)

In [ ]:
short = ["50건 초과 = 51건", "50건 초과 = 100건", "구간 하한으로 합산", "구간 상한으로 합산 (100 시나리오)", "순서 코드 합 (Φ=4, n=25%)",
         "원래 규칙 v1 (Φ=60.3, n=15%)", "하한 거의 없음 (Φ=1)",
         "약화 = 하위 10%", "약화 = 하위 2.5%", "추적 기간 3개월",
         "감소 기준 n = 25%", "감소 기준 n = 45%", "비교 기간 k = 3개월",
         "비교군 = 한 번도 신호 없던 기업", "층: 기준월을 합침", "층: 업종군 없이",
         "35개월 법인만 (독립 검증)", "35개월 이상 전체 (P35+)", "2024년 기준월만", "2025년 기준월만"]
titles = {"구간 가정": "① 구간→건수 변환 가정", "활동량 하한": "② 활동량 하한", "입금 약화 정의": "③ 약화(결과) 정의", "n·k": "④ 감소 기준·비교 기간", "비교군·층": "⑤ 비교 기업·층", "표본·기간": "⑥ 표본·기간"}
fig, ax = plt.subplots(figsize=(12.2, 9.6))
y, ys, labs = 0, [], []
band = True
last_group = None
rows_y = []
for v, sh in zip(VARIANTS, short):
    if v[0] != last_group:
        y -= 0.75 if last_group is not None else 0
        ax.text(-0.1, y, titles[v[0]], ha="right", va="center", fontsize=10.5, fontweight="bold", color=INK, transform=ax.get_yaxis_transform())
        y -= 0.65; last_group = v[0]; band = not band
    rows_y.append((y, v, sh, band)); y -= 1
lo_all = min(v[2] for v in VARIANTS); hi_all = max(v[2] for v in VARIANTS)
for yy, v, sh, band in rows_y:
    col = ORANGE if v[2] in (lo_all, hi_all) else BLUE
    ax.plot([v[3], v[4]], [yy, yy], color=col, lw=2.3, solid_capstyle="round", zorder=2); ax.plot([v[2]], [yy], "o", color=col, ms=8.5, mec="white", mew=1.2, zorder=3)
    ax.text(10.15, yy, fmt_ci(v[2], v[3], v[4]), va="center", fontsize=9.3)
    ax.text(-0.1, yy, sh, ha="right", va="center", fontsize=9.6, transform=ax.get_yaxis_transform())
    if band:
        ax.axhspan(yy - 0.5, yy + 0.5, color="#F4F6F9", zorder=0)
ax.axvline(1, color=GRAY, ls="--", lw=1.1); ax.axvline(VARIANT_BASE["rr"], color=ORANGE, ls=(0, (4, 3)), lw=1.2)
ax.text(VARIANT_BASE["rr"], 0.9, f"기준 RR {VARIANT_BASE['rr']}", color=ORANGE, ha="center", fontsize=9.5, fontweight="bold")
ax.set_xlim(0, 10); ax.set_ylim(y + 0.2, 1.4); ax.set_yticks([]); ax.grid(axis="y", visible=False); ax.spines["left"].set_visible(False)
ax.set_xlabel("상대위험도(RR)  ·  점 = 추정값, 선 = 95% 신뢰구간  ·  주황 점 = 20가지 중 가장 작은 값과 가장 큰 값")
ax.set_title("20가지 변형 모두 신뢰구간 하한이 1보다 크고, RR은 2.75 ~ 6.47에서 움직인다")
fig.subplots_adjust(left=0.36, right=0.88, top=0.94, bottom=0.07)
src(fig, "집계 결과 · 노트북 07 (variants) · 회색 점선 = RR 1(연관 없음), 주황 점선 = 기준 RR(4.69) · 변형 행의 기준값은 같은 코드로 다시 계산해 신뢰구간이 노트북 05와 소수 둘째 자리에서 다를 수 있음")
plt.show()
lo_v = min(VARIANTS, key=lambda v: v[2]); hi_v = max(VARIANTS, key=lambda v: v[2])
print(f"신뢰구간 하한이 1 이하인 변형: {sum(1 for v in VARIANTS if v[3] <= 1)}개 / 20개   |   RR 최소 {lo_v[2]} ({lo_v[1]})   RR 최대 {hi_v[2]} ({hi_v[1]})")

**읽는 법과 해석**

- **20가지 모두 신뢰구간 하한이 1보다 큼** — 연관이 사라지는 선택은 없었음. RR은 **2.75 ~ 6.47** 범위임.
- **가장 약한 변형은 '비교 기간 3개월'(RR 2.75)**: 짧은 비교 기간은 일시적으로 줄었다 회복하는 사례를 많이 포함하기 때문임(4장의 일시적 하락 비율 3개월 43% vs 6개월 25%).
- **약화 정의를 엄격하게 할수록 RR이 커짐**(하위 10% → 5% → 2.5%: 3.62 → 4.69 → 6.47). 그러나 이는 **결과(약화)의 정의 자체가 바뀐 것**이라 신호의 질을 비교한 것이 아니며, 기저 효과도 같은 패턴을 만들 수 있어 **증거로 쓰지 않고 관찰로만 기록**했음.
- 감소 기준 25% → 35% → 45%에서 RR이 3.84 → 4.69 → 4.71임. 35%는 요구불입금액을 보기 전에 건수 분포의 5백분위수로 정한 값이며, 이 비교는 그 선택을 사후에 뒷받침하는 정도임.
- **임시 기준을 30%로 정했던 비교 방식·비교 기간을 최종 기준 35%로 다시 선택해도 결정이 같았음**(직전 기간 대비, 6개월).
- **독립이 아님**: 20가지는 모두 같은 자료·같은 기업을 공유한 변형임(독립 검증 표본은 그중 하나).

### RR이 비슷해도 '잡히는 기업'은 다를 수 있다

변형마다 신호가 잡히는 **기업 집합**이 달라질 수 있음. 두 규칙이 **같은 기업을 최초 신호로 잡는 정도**를 **자카드 일치도(Jaccard)** 로 봄 — 두 집합의 교집합 ÷ 합집합이며, 1이면 완전히 같고 0이면 전혀 겹치지 않음.

(안씀)

In [ ]:
vj = [(sh, v) for sh, v in zip(short, VARIANTS) if v[6] is not None]
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.0), sharey=True, gridspec_kw={"width_ratios": [1, 1]})
yy = np.arange(len(vj))[::-1]
ax = axes[0]
ax.plot([v[2] for _, v in vj], yy, "o", color=BLUE, ms=9, mec="white"); ax.axvline(VARIANT_BASE["rr"], color=ORANGE, ls="--", lw=1.1)
for y_, (_, v) in zip(yy, vj):
    ax.text(v[2] + 0.22, y_, f"{v[2]:.2f}", va="center", fontsize=9.3)
ax.set_xlim(0, 8); ax.set_xlabel("RR (점추정)"); ax.set_yticks(yy); ax.set_yticklabels([sh for sh, _ in vj], fontsize=9.3); ax.grid(axis="y", visible=False)
ax.set_title("연관의 크기(RR)")
ax = axes[1]
ax.barh(yy, [v[6] for _, v in vj], color=[BLUE if v[6] >= 0.7 else PALE_ORANGE for _, v in vj], height=0.6, edgecolor=[BLUE if v[6] >= 0.7 else ORANGE for _, v in vj])
for y_, (_, v) in zip(yy, vj):
    ax.text(v[6] + 0.02, y_, f"{v[6]:.2f}", va="center", fontsize=9.3)
ax.set_xlim(0, 1.15); ax.set_xlabel("기준 규칙과 최초 신호 기업의 일치도 (Jaccard, 1 = 같은 기업)"); ax.grid(axis="y", visible=False)
ax.set_title("잡히는 기업의 구성")
fig.subplots_adjust(left=0.27, wspace=0.08)
finish(fig, "집계 결과 · 노트북 07 (variants) · 약화 정의를 바꾼 변형은 신호 기업이 같으므로 일치도가 1 · 일치도가 정의되지 않는 변형(비교 기업·층, 표본·기간)은 표시하지 않음", rect=(0.03, 0.04, 1, 1))

**읽는 법**: 왼쪽은 RR이 3~6 사이로 비슷한데, 오른쪽을 보면 **신호가 잡히는 기업의 구성은 변형에 따라 크게 달라짐**(주황 막대는 일치도 0.7 미만). 예를 들어 **원래 규칙(v1)** 은 일치도가 0.09뿐이고, **순서 코드 합**은 0.48임.

- 즉 '연관이 있다'는 큰 결론은 견고하지만, **어떤 기업이 신호로 잡히는지는 규칙에 따라 달라짐**. 신호 규칙이 '정답 기업 목록'을 주는 것이 아니라 **'위험이 더 높은 쪽'을 골라 주는 선별 기준**임을 보여 주는 결과임.

### 8장의 결론

- **연관은 분석 선택에 민감하지 않음**(20가지 모두 하한 > 1, RR 2.75~6.47). 단, **크기와 신호 기업의 구성은 선택에 따라 달라짐**.
- 가장 큰 영향은 **비교 기간**(3개월에서 가장 약함)과 **약화 정의**(엄격할수록 커짐)였음.

---
# 9장. 보강 점검 — 얼마나 맞히고 놓치나, 기준 기간 반론은 (노트북 08)

발표에서 받을 수 있는 두 가지 질문을 추가로 점검했음. **기준 규칙과 결론은 바꾸지 않았고**, 결과를 본 뒤 추가한 점검임.

1. 신호는 **얼마나 맞히고 얼마나 놓치는가**?
2. 신호와 약화가 **같은 기준 기간**을 쓰니 당연히 함께 움직이는 것 아닌가?

### 9.1 적중과 포착 — 신호 × 약화 2×2 표

신호 여부와 약화 여부로 기업-월을 네 칸으로 나눔.

> **통계 용어 — 적중(precision, 정밀도)과 포착(recall, 재현율·민감도)**
> - **적중 비율**(precision) = 신호가 잡힌 기업-월 중 **실제로 약화한 비율** = a ÷ (a + b). '신호가 얼마나 정확한가'.
> - **포착 비율**(recall, sensitivity) = 약화한 기업-월 중 **신호가 있었던 비율** = a ÷ (a + c). '약화를 얼마나 놓치지 않는가'.
> 두 값은 **서로 다른 질문**에 답하며, 기준을 엄격히 하면 적중은 오르고 포착은 내려가는 **상충 관계(trade-off)** 가 있음.


(검증하고 사용)

In [ ]:
a, b, c, d = TWO_BY_TWO["a"], TWO_BY_TWO["b"], TWO_BY_TWO["c"], TWO_BY_TWO["d"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [1.15, 1]})
ax = axes[0]; ax.set_xlim(-1.3, 3.5); ax.set_ylim(-1.2, 2.4); ax.axis("off"); ax.grid(False)
cells = [(0, 1, a, "신호이면서 약화", BLUE, "white"), (1, 1, b, "신호이면서 비약화", PALE_BLUE, INK), (0, 0, c, "비신호이면서 약화", ORANGE, "white"), (1, 0, d, "비신호이면서 비약화", LIGHT, INK)]
for cx, cy, n, name, fc, tc in cells:
    ax.add_patch(Rectangle((cx * 1.5, cy * 1.0), 1.45, 0.95, facecolor=fc, edgecolor="white", lw=2))
    ax.text(cx * 1.5 + 0.725, cy * 1.0 + 0.62, f"{n:,}", ha="center", va="center", fontsize=17, fontweight="bold", color=tc)
    ax.text(cx * 1.5 + 0.725, cy * 1.0 + 0.25, name, ha="center", va="center", fontsize=9.5, color=tc)
ax.text(0.725, 2.12, "약화", ha="center", fontsize=11, fontweight="bold"); ax.text(2.225, 2.12, "비약화", ha="center", fontsize=11, fontweight="bold")
ax.text(-0.15, 1.48, "신호\n기업-월", ha="right", va="center", fontsize=10.5, fontweight="bold"); ax.text(-0.15, 0.48, "비신호\n기업-월", ha="right", va="center", fontsize=10.5, fontweight="bold")
ax.text(3.1, 1.48, f"적중 {a / (a + b) * 100:.1f}%\n= {a} ÷ {a + b}", ha="left", va="center", fontsize=10.5, color=BLUE, fontweight="bold")
ax.text(3.1, 0.48, f"{c / (c + d) * 100:.1f}%\n= {c:,} ÷ {c + d:,}\n(조정 전)", ha="left", va="center", fontsize=9.5, color=MUTED)
ax.text(0.725, -0.2, f"포착 {a / (a + c) * 100:.1f}%\n= {a} ÷ {a + c:,}", ha="center", va="center", fontsize=10.5, color=ORANGE, fontweight="bold")
ax.set_title("기업-월 2×2 표 (P36, 13개 기준월)")

ax = axes[1]
h = HIT_P36; h5 = HIT_P35
labs = ["적중\n(신호 중 약화)", "포착\n(약화 중 신호)", "비신호의 약화율\n(조정 전)"]
v36 = [h["precision"], h["recall"], h["ctrl_raw"]]; v35 = [h5["precision"], h5["recall"], h5["ctrl_raw"]]
x = np.arange(3); w = 0.38
ax.bar(x - w / 2, v36, w, color=BLUE, label="P36 (규칙 결정)"); ax.bar(x + w / 2, v35, w, color=PURPLE, label="P35만 (독립 검증)")
for i in range(3):
    ax.text(i - w / 2, v36[i] + 0.7, f"{v36[i]:.1f}%", ha="center", fontsize=10); ax.text(i + w / 2, v35[i] + 0.7, f"{v35[i]:.1f}%", ha="center", fontsize=10)
ax.set_xticks(x); ax.set_xticklabels(labs, fontsize=9.8); ax.set_ylim(0, 38); ax.set_ylabel("%"); ax.legend(loc="upper right"); ax.grid(axis="x", visible=False)
ax.set_title("적중은 24.6%, 포착은 10.8%")
finish(fig, "집계 결과 · 노트북 08 (two_by_two, hit_summary) · 기업-월 단위라 같은 기업이 여러 기준월에 세어질 수 있음 · 층화·표준화 비교 기업의 약화율은 5.3%")

**읽는 법과 해석**

- **적중 24.6%**: 신호가 잡힌 487건 중 120건만 실제 약화임 → 신호 기업의 약 **75%는 약화가 아님**. 다만 비신호 기업의 약화율(3.6~5.3%)보다는 약 4.7배 높음 — 이것이 RR 4.69임.
- **포착 10.8%**: 약화한 1,109건 중 **그 달에 신호가 있었던 것은 120건뿐**임 → 약화의 **약 89%는 신호 없이** 일어남.
- 신호는 약화를 모두 잡아내는 도구가 아니라 **상담 우선순위를 정하는 선별 도구**임. 아래 그림은 월평균 규모로 본 같은 이야기임.

(안씀)

In [ ]:
a, c = TWO_BY_TWO["a"], TWO_BY_TWO["c"]; nm = N_GRID[0]["n_months"]
weak_pm, caught_pm, sig_pm = (a + c) / nm, a / nm, next(x for x in N_GRID if x["n"] == 35)["n_detected"] / nm     # 월평균 신호 기업 = 최초 신호 기업 수 ÷ 13
cells, caught = int(round(weak_pm)), int(round(caught_pm))
cols = 17
fig, ax = plt.subplots(figsize=(11, 3.4))
for i in range(cells):
    r_, c_ = divmod(i, cols)
    ax.add_patch(Rectangle((c_ * 1.1, -r_ * 1.1), 1, 1, facecolor=BLUE if i < caught else LIGHT, edgecolor="white"))
ax.set_xlim(-0.2, cols * 1.1 + 12); ax.set_ylim(-5.6, 1.4); ax.set_aspect("equal"); ax.axis("off"); ax.grid(False)
ax.text(cols * 1.1 + 0.8, -0.8, f"월평균 약화 기업-월 약 {weak_pm:.0f}건 중", fontsize=11.5, fontweight="bold", va="center")
ax.text(cols * 1.1 + 0.8, -1.9, f"■ 신호가 그 달에 잡은 것 약 {caught_pm:.0f}건 ({caught_pm / weak_pm * 100:.1f}%)", fontsize=11, color=BLUE, va="center", fontweight="bold")
ax.text(cols * 1.1 + 0.8, -3.0, f"■ 신호 없이 일어난 약화 약 {weak_pm - caught_pm:.0f}건", fontsize=11, color=GRAY, va="center", fontweight="bold")
ax.text(cols * 1.1 + 0.8, -4.3, f"(월평균 신호 기업은 약 {sig_pm:.0f}곳)", fontsize=10, color=MUTED, va="center")
ax.set_title("정사각형 1칸 = 약화 기업-월 1건 (월평균)", loc="left")
finish(fig, "집계 결과 · 노트북 08 (hit_summary) · 13개 기준월 평균 · 기업-월 단위")

### 9.2 기업 단위로 보면 — 신호가 약화보다 먼저 잡히나

기업-월 단위는 같은 기업이 여러 달 세어지므로, **기업 단위**로도 봄. **약화 기업**은 판정 가능 기준월 중 **한 번이라도 약화**한 기업이고, 그 **첫 약화 시점과 신호 시점**을 비교함.

- **신호가 먼저(또는 같은 달)**: 신호가 약화보다 앞서거나 같은 달에 잡힘
- **신호가 늦음**: 약화가 먼저이고 신호는 그 뒤에 잡힘
- **신호가 없음(놓침)**: 신호가 한 번도 잡히지 않음

(안씀)

In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 3.3))
rows = [("36개월 법인 (P36)", TIMING["P36"]), ("35개월 법인만 (독립 검증)", TIMING["P35"])]
for i, (nm, t) in enumerate(rows[::-1]):
    parts = [t["early"], t["late"], t["none"]]; tot = t["n_weak"]
    left = 0
    for v, col, lab in zip(parts, [BLUE, PALE_BLUE, ORANGE], ["신호가 먼저", "신호가 늦음", "신호 없음(놓침)"]):
        pct = v / tot * 100
        ax.barh(i, pct, left=left, color=col, height=0.55, edgecolor="white")
        ax.text(left + pct / 2, i, f"{lab}\n{pct:.1f}% ({v}곳)", ha="center", va="center", fontsize=9.6, color="white" if col in (BLUE, ORANGE) else INK, fontweight="bold")
        left += pct
    ax.text(101, i, f"약화 기업 {tot}곳", va="center", fontsize=10)
ax.set_yticks([0, 1]); ax.set_yticklabels([r[0] for r in rows[::-1]]); ax.set_xlim(0, 118); ax.set_xticks([0, 25, 50, 75, 100]); ax.grid(axis="y", visible=False)
ax.set_xlabel("약화 기업 중 비율(%)")
ax.set_title("약화 기업의 44%는 신호가 한 번도 없었고, 28%는 약화보다 늦게 잡혔다")
finish(fig, "집계 결과 · 노트북 08 (firm_timing) · 약화 기업 = 판정 가능 기준월 중 한 번이라도 약화한 기업(36개월 법인 2,511곳 중 371곳, 35개월 법인만 978곳 중 224곳)")

**읽는 법과 해석**

- 약화 기업 중 **신호가 먼저(또는 같은 달) 잡힌 기업은 27.5%**, **더 늦게 잡힌 기업은 28.3%**, **한 번도 잡히지 않은 기업은 44.2%** (36개월 법인).
- 신호가 **항상 약화보다 먼저 잡히지 않음** → 앞서(6.5절) '조기경보가 아닌 함께 나타나는 신호'로 해석한 것과 일치함.
- 약화 기업의 상당수는 **건수 신호로는 잡히지 않음**. 건수가 유지된 채 요구불입금액만 줄어드는 경우 등이 있을 수 있음(해석 가설). **요구불입금액 기반 신호와의 비교·결합이 필요한 이유**임(후속 과제).

### 9.3 감소 기준 n을 바꾸면 — 적중과 포착의 상충

감소 기준 n만 바꾸고 나머지(활동량 하한·비교 기간·약화 기준·규모 경계)는 **P36에서 정한 값으로 고정**해 다시 계산했음.


(씀)

In [ ]:
g = N_GRID
n_ = [x["n"] for x in g]
prec = [x["precision"] * 100 for x in g]; rec = [x["recall"] * 100 for x in g]; nsig = [x["n_detected"] / x["n_months"] for x in g]; rr_ = [x["rr"] for x in g]
fig, axes = plt.subplots(1, 3, figsize=(14, 4.3))
ax = axes[0]
ax.plot(n_, prec, color=BLUE, marker="o", lw=2.2, label="적중 (신호 중 약화)"); ax.plot(n_, rec, color=ORANGE, marker="s", lw=2.2, label="포착 (약화 중 신호)")
ax.text(n_[-1] + 1, prec[-1], f"{prec[-1]:.0f}%", color=BLUE, va="center", fontsize=9.5); ax.text(n_[-1] + 1, rec[-1], f"{rec[-1]:.0f}%", color=ORANGE, va="center", fontsize=9.5)
ax.axvline(35, color=GRAY, ls="--", lw=1.1); ax.text(35.8, 3.0, "기본 35%", color=MUTED, fontsize=9.5)
ax.set_xlabel("감소 기준 n (%)"); ax.set_ylabel("%"); ax.set_ylim(0, 58); ax.legend(loc="upper left", fontsize=9.3); ax.set_xlim(17, 76)
ax.set_title("적중은 오르고 포착은 내린다")
ax = axes[1]
ax.bar(range(len(n_)), nsig, color=[BLUE if v == 35 else PALE_BLUE for v in n_], width=0.65)
for i, v in enumerate(nsig):
    ax.text(i, v + 1.5, f"{v:.0f}", ha="center", fontsize=9.5)
ax.set_xticks(range(len(n_))); ax.set_xticklabels([f"{int(v)}" for v in n_]); ax.set_xlabel("감소 기준 n (%)"); ax.set_ylabel("월평균 신호 기업 수(곳)"); ax.grid(axis="x", visible=False)
ax.set_title("상담 대상은 줄어든다")
ax = axes[2]
ax.plot(n_, rr_, color=PURPLE, marker="o", lw=2.2)
for x_, v in zip(n_, rr_):
    ax.text(x_, v + 0.35, f"{v:.1f}", ha="center", fontsize=9)
ax.axhline(1, color=GRAY, ls="--", lw=1); ax.set_ylim(0, 9); ax.set_xlabel("감소 기준 n (%)"); ax.set_ylabel("상대위험도 RR(점추정)")
ax.set_title("RR은 기준이 엄격할수록 커진다")
finish(fig, "집계 결과 · 노트북 08 (n_grid) · 13개 기준월 평균, 기업-월 단위 · 월평균 신호 기업 = 최초 신호 기업 수 ÷ 13")

**읽는 법과 해석**

- 기준을 **엄격히(n↑)** 하면 신호가 적어져 **적중은 오르고 포착은 내려감**(35%: 적중 24.6%·포착 10.8% → 60%: 적중 42.0%·포착 3.9%). 기준을 **느슨히(n↓)** 하면 포착은 오르지만 적중이 15.0%(20%)까지 내려감.
- **어느 기준이 '옳은가'는 상담 인력(운영 용량)과 비용으로 정하는 문제**임. 이 분석은 기본을 **35%**(건수 분포의 5백분위수 근거)로 두고, 인력이 부족하면 45~50% 이상을 우선 상담 대상으로 두는 것을 제안함(10장).
- RR이 기준에 따라 3.6~7.6으로 커지는 것은 비교 기업의 약화율이 4.2~6.8%로 거의 일정한 채 신호 기업의 약화율이 높아지기 때문임. 단, **엄격한 기준에서는 신호 기업 수가 적어 불확실성이 큼**(60%에서는 월평균 10곳).

### 9.4 기준 기간 반론 — 신호와 약화가 같은 달을 비교 기준으로 쓰는 것 아닌가

기준 규칙에서는 **신호의 비교 기간(직전 6개월)** 과 **약화의 기준 기간(전년 동기)** 이 **같은 6개월**임(신호 시점 기준 11~6개월 전). 그 기간의 일시적 호조가 건수와 요구불입금액을 함께 높였다면, 평균 회귀만으로 신호와 약화가 함께 나타날 수 있음(5.3절의 합성 예시).

**점검 방법**: 신호를 **'전년 같은 달 대비 감소'** 로 다시 정의함. 이렇게 하면 신호의 비교 기간이 **17~12개월 전**으로 옮겨 가 **약화의 기준 기간과 겹치지 않음.** 이 규칙의 활동량 하한(11.1)과 감소 기준(45%)은 **건수만으로 같은 절차로 다시 구했음.**

(안씀)

In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 3.6))
ax.set_xlim(-18.6, 7.4); ax.set_ylim(-0.3, 4.1); ax.grid(False); ax.spines["left"].set_visible(False); ax.set_yticks([])
ax.set_xticks(range(-17, 7)); ax.set_xticklabels([f"{m:+d}" if m else "e" for m in range(-17, 7)], fontsize=8.5)
ax.set_xlabel("신호 기준월 e = 0 으로 둔 상대 월")
def bar(y, a, b, color, label, hatch=None, tc="white", alpha=1.0):
    ax.add_patch(Rectangle((a - 0.5, y - 0.3), b - a + 1, 0.6, facecolor=color, edgecolor="white", lw=1.2, hatch=hatch, alpha=alpha, zorder=2))
    ax.text((a + b) / 2, y, label, ha="center", va="center", fontsize=9.3, color=tc, fontweight="bold", zorder=3)
ax.axvspan(-11.5, -5.5, color=PALE_ORANGE, alpha=0.4, zorder=0)
ax.text(-18.4, 3.55, "[기준 규칙] 직전 기간 대비", fontsize=10, fontweight="bold", color=BLUE, va="center")
bar(3.0, -11, -6, GRAY, "신호의 비교 달")
bar(2.35, -11, -6, GRAY, "약화의 기준 달(전년 동기)", hatch="///", tc=INK, alpha=0.55)
ax.text(-2.5, 2.7, "6개월 겹침", fontsize=11, fontweight="bold", color=ORANGE, va="center")
ax.text(-18.4, 1.55, "[이번 점검] 전년 같은 달 대비", fontsize=10, fontweight="bold", color=BLUE, va="center")
bar(1.0, -17, -12, GRAY, "신호의 비교 달 (전년 같은 달)")
bar(0.35, -11, -6, GRAY, "약화의 기준 달(전년 동기)", hatch="///", tc=INK, alpha=0.55)
ax.text(-2.5, 0.68, "0개월 겹침", fontsize=11, fontweight="bold", color=BLUE, va="center")
ax.set_title("기준 기간을 겹치지 않게 만든 신호로 한 번 더 확인한다")
finish(fig, "개념도 · 집계 결과 · 노트북 08 (reference_months): 겹치는 달 수 6개월 vs 0개월")

In [ ]:
Y = YOY
rows = [("36개월 법인 · 기준 규칙\n(직전 기간 대비)", Y["P36_base"]["rr"], Y["P36_base"]["lo"], Y["P36_base"]["hi"], BLUE),
        ("36개월 법인 · 전년 같은 달 대비\n(기준 달 비공유)", Y["P36_yoy"]["rr"], Y["P36_yoy"]["lo"], Y["P36_yoy"]["hi"], ORANGE),
        ("35개월 법인만 · 기준 규칙", Y["P35_base"]["rr"], Y["P35_base"]["lo"], Y["P35_base"]["hi"], PURPLE),
        ("35개월 법인만 · 전년 같은 달 대비", Y["P35_yoy"]["rr"], Y["P35_yoy"]["lo"], Y["P35_yoy"]["hi"], ORANGE)]
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.3), gridspec_kw={"width_ratios": [1.7, 1]})
ax = axes[0]
forest(ax, rows, xmax=8, base=None, label_fs=9.5)
ax.set_title("기준 달을 공유하지 않아도 RR이 4.0 이상으로 유지된다")
ax = axes[1]
rr_rows = [("36개월 법인\n(전년 같은 달 ÷ 기준 규칙)", *Y["ratio_P36"], ORANGE), ("35개월 법인만", *Y["ratio_P35"], ORANGE)]
forest(ax, rr_rows, xmax=2.0, ref=1.0, xlabel="RR의 비 · 1 = 같음", label_fs=9.5)
ax.set_title("두 신호 정의의 RR은 구별되지 않는다")
fig.subplots_adjust(wspace=0.95, left=0.22, right=0.92)
finish(fig, "집계 결과 · 노트북 08 (year_on_year) · 기준 규칙 행의 신뢰구간은 같은 재표집으로 다시 계산한 값이라 노트북 05의 3.75~5.95와 소수 둘째 자리가 약간 다름")
tab = pd.DataFrame(TWO_RULES_HIT, columns=["신호 정의", "적중 비율(%)", "포착 비율(%)", "월평균 신호 기업(곳)"])
display(tab.set_index("신호 정의"))

**읽는 법과 해석**

- 전년 같은 달 대비로 정의한 신호에서도 **RR 4.36 (3.39~5.60)** 으로 신뢰구간 하한이 1을 크게 넘었고, 독립 표본에서도 **RR 4.02 (3.07~5.52)** 임.
- 두 정의의 **RR의 비**는 **0.93 (0.75~1.16)** 으로 **1을 포함** → 기준 규칙과 구별되지 않음. 최초 신호 기업의 일치도(Jaccard)는 0.49로 신호가 잡히는 기업은 상당히 다르지만, 연관의 크기는 비슷함.
- 따라서 **기준 달을 공유한다는 이유만으로 연관이 만들어진 것은 아님.** 합성 예시(5.3절)와 달리 실제 자료에서는 연관이 **사라지지 않았음**.
- **다만 같은 요구불입금액 자료를 쓰므로 평균 회귀·기저 효과의 모든 경로를 배제한 것은 아님.** 이 점검은 결과를 본 뒤 추가한 보강 점검임.
- 새 신호 정의의 적중은 25.7%로 비슷하고, 포착은 7.3%, 월평균 신호 기업은 약 26곳임.

### 9장의 결론

| 질문 | 답 |
|---|---|
| 얼마나 맞히나(적중) | 신호 기업-월의 **24.6%** 가 약화 (비신호는 3.6~5.3%) |
| 얼마나 놓치나(포착) | 약화 기업-월의 **10.8%** 에서만 그 달에 신호가 있음. 약화 기업의 **44%는 신호가 한 번도 없음** |
| 신호가 먼저 잡히나 | 먼저 27.5%, 늦게 28.3%, 없음 44.2% → **조기경보 아님** |
| 기준 기간 공유 반론 | 비공유 신호로 다시 정의해도 **RR 4.36(독립 4.02)**, 기준 규칙과 구별되지 않음. 모든 경로를 배제한 것은 아님 |

---
# 10장. 활용 방안 — 신호가 잡히면 무엇을 하나

이 장은 **분석 결과를 현업에서 어떻게 쓸 수 있는가**를 제안함. **처방의 효과는 검증하지 않았음.** 아래는 상담에서 확인할 **가설과 후보**이며, **근거 등급**으로 데이터가 말해 주는 것과 업무 가정을 구분했음.

### 10.1 운영 시나리오

In [ ]:
steps = [("① 월 단위 신호 산출", "판정 가능 기업의\n국내 5채널 건수 합이\n직전 6개월 대비\n35% 이상 감소", "이번 분석 결과"),
         ("② 신호 기업 목록", "월평균 약 38곳\n(위험집합의 약 1.8%)\n월평균 약화 85건 중\n약 9건 포착", "이번 분석 결과"),
         ("③ 상담에서 원인 확인", "주거래 이전·사업 축소\n일시 감소·채널 이동\n중 무엇인지 질문", "운영 제안"),
         ("④ 처방 후보 적용", "원인별 대응\n(10.3 표)", "운영 제안 · 효과 미검증"),
         ("⑤ 효과 추적(파일럿)", "신호 기업 중 일부만\n적용하고 미적용과 비교", "후속 과제")]
fig, ax = plt.subplots(figsize=(13.5, 3.7))
ax.set_xlim(0, 13.6); ax.set_ylim(0, 3.7); ax.axis("off"); ax.grid(False)
for i, (t, d, tag) in enumerate(steps):
    x0 = 0.1 + i * 2.72
    result = tag == "이번 분석 결과"
    fc = BLUE if result else "white"; ec = BLUE if result else ORANGE; tc = "white" if result else INK
    ax.add_patch(FancyBboxPatch((x0, 0.55), 2.35, 2.4, boxstyle="round,pad=0.02,rounding_size=0.12", facecolor=fc, edgecolor=ec, lw=2, ls="-" if result else (0, (4, 3))))
    ax.text(x0 + 1.175, 2.62, t, ha="center", va="center", fontsize=10.8, fontweight="bold", color=tc)
    ax.text(x0 + 1.175, 1.6, d, ha="center", va="center", fontsize=9.4, color=tc, linespacing=1.5)
    ax.text(x0 + 1.175, 0.28, tag, ha="center", va="center", fontsize=9.2, color=BLUE if result else ORANGE, fontweight="bold")
    if i < 4:
        ax.annotate("", xy=(x0 + 2.69, 1.75), xytext=(x0 + 2.4, 1.75), arrowprops=dict(arrowstyle="-|>", color=GRAY, lw=1.8))
ax.set_title("신호에서 처방까지: 파랑 = 이번 분석에서 확인한 단계, 주황 점선 = 제안·미검증 단계")
finish(fig, "개념도 · 월평균 신호 기업과 약화 기업-월 수는 노트북 05·08의 13개 기준월 평균(기업-월 단위)", rect=(0, 0.03, 1, 1))

### 10.2 감소 기준별 대상 규모 — 인력에 맞춰 고르기

9.3절의 결과를 운영 관점에서 정리한 표임(13개 기준월 평균, 적중·포착은 기업-월 단위).

In [ ]:
tab = pd.DataFrame([{"감소 기준 n": f"{int(x['n'])}%" + (" (기본)" if x["n"] == 35 else ""), "월평균 신호 기업(곳)": round(x["n_detected"] / x["n_months"]),
                     "적중: 신호 중 약화(%)": round(x["precision"] * 100, 1), "포착: 약화 중 신호(%)": round(x["recall"] * 100, 1), "RR": round(x["rr"], 1)} for x in N_GRID])
display(tab.set_index("감소 기준 n"))

- 기준을 높일수록 대상은 줄고 적중은 오르지만 포착은 내려감. **기본은 35%로 넓게 감시**하고, **인력이 부족하면 45~50% 이상을 우선 상담 대상**으로 두는 것을 제안함. (영업점 배분 인력은 가정임.)
- 월평균 신호 기업 수는 최초 신호 기업 수를 13개 기준월로 나눈 값이며, 월별 위험집합(약 2,100곳)의 약 1.8%임(근사).

### 10.3 신호 → 가능한 원인 → 처방 후보

> **근거 등급** — **A** 데이터 지지(재현·강건) · **B** 데이터와 비모순(방향만 일치·탐색적) · **C** 도메인 가정(이 데이터로 확인 불가)

| 고객·신호 | 데이터에서 보이는 모습 | 처방 후보 | 등급 | 등급의 근거 |
|---|---|---|---|---|
| **신호가 잡힌 기업** | 건수 감소 후 요구불입금액도 약화 | 유동성 지원 상담(운전자금, 이자 부담 완화), 사업 상황 확인 | A(신호)·C(원인) | 신호와 약화의 연관은 재현·강건함. **감소 원인은 데이터로 구분 불가** → 상담 질문으로 확인 |
| **채널 이동** (총활동량 유지) | 오프라인 비중만 감소 | 합산 신호의 대상이 아님. **별도 이동 신호로 분류**해 원인을 확인한 뒤 채널 전환 이용 안내 | B | 이동 기업 37곳(약 1.5%), 약화는 4곳 이하(RR 미계산). 위험이 없다는 뜻은 아님 |
| **수주·계절형 일시 감소** (평균 회귀) | 일시적으로 줄었다 회복, 건설·프로젝트 업종, 신호 직전 요구불입금액 급등 | 즉시 개입 대신 **모니터링 연장**, 다음 수주 시점 확인 | B | 2024.12 이후 신호 기업의 약 27%가 직전 급등 기업(비교 기업 약 9%), 3개월 비교 기간에서 신호가 약해짐 |
| **관계가 얕은 기업** | 자동이체가 적고 평소 요구불입금액이 불안정(약화 비율 8.0% vs 3.9%) | **접점 확대**(복수 채널·자동이체 개설 안내) | B | 탐색적 |
| **자동이체가 많은 고객에서 신호** | 평소 안정적이라 이례적이나 신호 기업 약화율은 22%로 낮음 | **해석 보조 정보**로만 사용. 우선순위를 높이는 근거로는 쓰지 않음 | B | RR이 큰 것은 비교 기업의 약화가 드물어서이며 약한 단서 |
| **주거래 이전** (타행으로 대금 수취 이동) | 요구불입금액은 줄지만 자동이체·채널 이용은 유지 | 수납·결제 계좌 **재유치 혜택**, 자동이체 연계 제안 | C | 타행 이동은 이 데이터로 확인 불가 → 상담 질문 |
| **자동이체 이탈** | 자동이체 코드 하락 | 결제·수납 연계 재제안 | C | 이번 분석은 자동이체 '수준'만 보았고 하락 신호는 검증하지 않음 |
| **외환 거래 감소** | 외환 건수 감소(독립 축) | 외환 업무 별도 상담 | C | 이용 기업이 적고(약 8%) 업종 편중, 검증하지 않음 |
| **거래 활동량이 큰 고객** (건수 신호가 둔감) | 신호 발생 3%, 건수 구간 상한에 닿음 | **금액 기반 신호를 병행**하는 별도 관리 | A(사각지대)·C(처방) | 큰 규모에서 신호가 드문 것은 데이터로 확인 |
| **거의 거래가 없는 고객** | 활동량 하한 미만 805곳, 판정 불가 | **별도 휴면·저활동 관리** 후보 | A(사각지대)·C(처방) | 약화 비율 14.0%(판정 가능 기업 5.0%), 단 소액이라 상대 변화가 쉽게 커지는 효과 가능 |

- 구체적인 **상품명은 제안하지 않음.** 상품 보유 변수를 이번 분석에 쓰지 않았기 때문이며, 보유 상품·카드·여신 변수를 추가하면 신호 이전의 보유 상태를 이용해 **상품군 수준**의 처방 후보로 구체화할 수 있음.

### 10.4 사각지대 두 곳 — 건수 신호가 못 보는 고객

건수 신호가 판정하지 못하거나 둔감한 고객이 두 부류 있음.

(마지막 한계로 이쁘게 씀)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 3.9))
ax = axes[0]
ax.bar([0, 1], [FLOOR_EXCLUDED["weak_pct"], JUDGEABLE_FIRMS["weak_pct"]], color=[ORANGE, BLUE], width=0.55)
ax.text(0, FLOOR_EXCLUDED["weak_pct"] + 0.4, f"{FLOOR_EXCLUDED['weak_pct']:.1f}%", ha="center", fontsize=11, fontweight="bold")
ax.text(1, JUDGEABLE_FIRMS["weak_pct"] + 0.4, f"{JUDGEABLE_FIRMS['weak_pct']:.1f}%", ha="center", fontsize=11, fontweight="bold")
ax.set_xticks([0, 1]); ax.set_xticklabels([f"활동량 하한 미만\n(판정 불가) {FLOOR_EXCLUDED['n']}곳", f"판정 가능 기업\n{JUDGEABLE_FIRMS['n']:,}곳"])
ax.set_ylim(0, 18); ax.set_ylabel("요구불입금액 약화 비율(%)"); ax.grid(axis="x", visible=False)
ax.set_title("거의 거래가 없는 기업: 판정은 못 하지만 약화는 더 잦다")
ax = axes[1]
rate = [x[3] for x in SIZE_RR]
ax.bar(range(5), rate, color=[BLUE] * 4 + [ORANGE], width=0.6)
for i, v in enumerate(rate):
    ax.text(i, v + 1, f"{v:.1f}%", ha="center", fontsize=10.5, fontweight="bold" if i == 4 else None)
ax.set_xticks(range(5)); ax.set_xticklabels(["1분위\n(작음)", "2분위", "3분위", "4분위", "5분위\n(큼)"]); ax.set_ylim(0, 52); ax.set_ylabel("신호 발생 비율(%)"); ax.grid(axis="x", visible=False)
ax.set_title("거래 활동량이 큰 기업: 건수 신호가 잘 잡히지 않는다")
finish(fig, "집계 결과 · 노트북 05 (floor_excluded, heterogeneity) · 하한 미만 기업의 약화 비율은 소액이라 상대 변화가 쉽게 커지는 효과가 섞였을 수 있음(해석 가설)")

- **거의 거래가 없는 고객(805곳)**: 건수가 너무 적어 변화율이 의미가 없어 판정에서 제외했지만, 이 기업들의 요구불입금액 약화 비율은 **14.0%**로 판정 가능 기업(5.0%)보다 높았음. → **별도의 휴면·저활동 관리 후보**.
- **거래 활동량이 큰 고객**: 건수 구간 상한에 닿아 신호 발생이 3%에 그침. → **금액 기반 신호를 병행**하는 별도 관리가 필요함.

### 10.5 상담 시 확인 질문 (원인 판별용)

고객과의 대화에서 쓰는 문장이라 일상어(입금)를 그대로 썼음.

1. 대금을 받는 계좌를 다른 은행으로 바꾸셨나요? → *주거래 이전*
2. 최근 거래처·매출·사업 규모에 변화가 있나요? → *사업 축소*
3. 일회성 입금이 끝난 것인가요? 다음 수주·입금 시점은 언제인가요? → *수주형·일시 감소*
4. 인터넷·스마트뱅킹 등 이용 방식을 바꾸셨나요? → *채널 이동*
5. 자동이체로 처리하던 결제를 다른 방식으로 옮기셨나요? → *자동이체 이탈*

---
# 11장. 결론, 한계, 후속 과제

### 11.1 핵심 질문에 대한 답

| 질문 | 답 | 근거(장) |
|---|---|---|
| ① 건수 감소는 이후 요구불입금액 약화와 연결되는가 | **그렇다.** 신호 기업의 약화 비율 24.6%, 같은 구성의 비교 기업 5.3% → **RR 4.69 (3.75~5.95)** | 6장 |
| ② 어떤 고객에서 강하고 약한가 | 모든 규모·세 채널 유형에서 연관이 있고 유형 간 차이는 확인되지 않음. 단 **거래 활동량이 큰 고객은 신호가 드묾**(사각지대), 거의 거래가 없는 고객은 판정 불가 | 7장 |
| ③ 분석 선택을 바꿔도 같은가 | **연관은 유지된다.** 독립 표본 RR 4.29, 20가지 변형 RR 2.75~6.47(모두 하한 > 1), 기준 기간 비공유 신호 RR 4.36. 단 **크기와 신호 기업의 구성은 선택에 따라 달라진다** | 6·8·9장 |
| ④ 신호가 잡히면 어떻게 활용하는가 | 월평균 약 38곳을 **상담 우선순위**로 삼고 원인(주거래 이전·사업 축소·일시 감소·채널 이동)을 확인해 처방 후보를 적용. 효과는 파일럿으로 검증 필요 | 10장 |

### 11.2 말할 수 있는 것과 말하지 않는 것

| 말할 수 있는 것 | 말하지 않는 것 |
|---|---|
| 건수 감소 신호와 이후 요구불입금액 약화가 **함께 나타나는 경향**(RR 약 4.7) | 건수 감소가 요구불입금액 감소를 **일으켰다**는 인과 |
| 이 경향은 독립 표본·20가지 조건·세 채널 유형에서 **같은 방향** | 신호가 요구불입금액보다 **먼저** 움직이는 **조기경보** |
| 신호는 상담 우선순위를 정하는 **선별 도구** (같은 인력으로 약화 기업을 약 4.7배 더 만남) | 신호가 약화를 **모두 잡아낸다**는 것 (포착 10.8%, 약화 기업의 44%는 신호 없음) |
| 신호 이전에는 신호 기업의 요구불입금액이 오히려 높았음(평균 회귀 가능성) | 처방이 위축·이탈을 **줄인다**는 것 (미검증) |

### 11.3 한계

1. **연관이지 인과가 아님.** 건수가 줄어든 이유(매출 감소·타행 이동·채널 전환)는 이 데이터로 구분할 수 없고, 처방의 효과는 검증하지 않았음.
2. **요구불입금액 감소를 위축으로 본 것은 업무상 가정**임(대금 수취가 요구불 계좌로 들어오므로 사업 활동의 대용 지표로 봄, 타행 이동도 은행 입장에서는 같은 거래 감소). 이 가정은 이번 데이터로 검증할 수 없음.
3. **신호와 약화의 기준 기간 공유**: 기준 달을 공유하지 않는 신호로 재분석해도 유지되었으나 같은 자료를 쓰므로 모든 경로를 배제하지는 못함.
4. **건수는 구간 자료**: 간격이 고르지 않고 50건 초과는 한 구간이라 거래가 많은 기업은 변화를 둔하게 잡음(5분위 신호 발생 3%).
5. **요구불입금액은 반올림된 값**: 약화 경계 부근에서 오분류가 생길 수 있음(6개월 평균으로 완화, 금액 구간별 확인은 하지 않음).
6. **판정 가능 기업에 한정**: 36개월 법인의 약 25%(861곳)는 판정하지 않았고 그중 활동량 하한 미만이 805곳임.
7. **이탈이 아니라 존속 고객의 위축**: 35개월 이상 관측된 법인(전체의 약 31%)만 분석했고, **표본 선정 편향이 없다는 가정**에 의존함(팀 내 별도 검증).
8. **잔여 불균형**: 표준화 후에도 고객등급(최우수 비중)·전담 고객·지역·탐지 전 요구불입금액 추세의 차이(표준화 차이 0.11~0.18)가 남았으며 RR에 반영되지 않았음.
9. **독립 검증은 같은 기간의 다른 기업**: 시간적 독립 검증이 아니며, 기준월을 나누면 RR이 2024년 4.05, 2025년 5.78로 달라 시기별 변동이 있음. 신호는 사후에 고정한 규칙이라 실시간 성능이 아님.
10. **요구불입금액 기반 신호와 비교하지 않음**: 건수 신호가 금액으로 직접 보는 것보다 더 알려 주는 정보가 있는지는 확인되지 않음.
11. **탐색 결과는 약함**: 유형·자동이체·채널 이동 비교는 탐색이며 자동이체 차이는 신뢰구간 하한이 1에 가까움.

### 11.4 후속 과제

| 과제 | 필요한 이유 | 방법(안) |
|---|---|---|
| 요구불입금액 기반 신호와의 비교·결합 | 약화 기업의 44%는 건수 신호가 한 번도 없었음 | 두 신호의 겹침, 한쪽에서만 잡히는 기업의 약화 비율 비교(팀 내 분석과 연결) |
| 신호가 늦거나 없는 약화 기업의 특성 파악 | 약화 기업의 28%는 신호가 늦게, 44%는 한 번도 잡히지 않음 | 채널 이용·자동이체·업종·활동량을 신호가 먼저 잡힌 기업과 비교 |
| 처방 효과 파일럿 | 처방이 위축·이탈을 줄이는지 검증하지 않음 | 신호 기업 중 일부에만 상담을 적용하고 미적용 기업과 비교 |
| 이후 기간 데이터로 검증 | 같은 기간의 다른 기업만 확인했고 규칙은 사후에 고정함 | 2026년 데이터에 고정 규칙을 그대로 적용 |
| 금액 변수 병행 | 거래 활동량이 큰 고객은 건수 구간 상한 때문에 신호가 둔함 | 채널별 거래금액 등 상한이 없는 변수로 보완 |
| 상품·카드·여신 변수 추가 | 처방 후보를 상품군 수준으로 구체화하지 못함 | 신호 이전 시점의 보유 상태를 추가로 불러와 분석 |
| 관측 개월이 짧은 법인의 구성 확인 | 이탈·신규·공백 중 무엇인지 몰라 이탈 해석에 한계 | 제외된 법인의 관측 패턴 요약 |

### 11.5 한 문장 결론

> 거래 건수가 직전 6개월 대비 35% 이상 감소한 기업은 이후 6개월 요구불입금액이 크게 약화될 가능성이 비교 기업의 약 4.7배였고, 이는 독립 표본·20가지 분석 조건·세 채널 유형에서 같은 방향으로 확인되었음. 다만 약화 기업의 일부만 포착하는 **선별 도구**이므로, 신호가 잡히면 상담으로 원인을 확인하고 거래 활동량이 큰 고객과 거래가 거의 없는 고객은 별도 신호로 관리하는 것을 제안함.

---
# 부록 A. 통계 용어 사전

| 용어 | 뜻 | 이 분석에서 |
|---|---|---|
| **상대위험도(RR, relative risk)** | 두 집단에서 어떤 사건이 일어나는 **비율의 비**. 1이면 차이 없음 | 신호 기업의 약화 비율 ÷ 비교 기업의 약화 비율(관찰 ÷ 기대) |
| **점추정(point estimate)** | 표본에서 계산한 하나의 값 | RR 4.69 |
| **95% 신뢰구간(CI)** | 표본이 달라질 때 추정값이 움직일 수 있는 범위. 같은 방법을 반복하면 구간이 진짜 값을 포함하는 경우가 약 95% | RR 3.75~5.95. **하한 > 1이면 채택** |
| **부트스트랩(bootstrap)** | 가진 표본에서 **복원추출**로 가짜 표본을 많이 만들어 추정값의 흔들림을 재는 방법 | 2,000회 반복, 2.5~97.5 백분위수가 95% 신뢰구간 |
| **군집 부트스트랩(cluster bootstrap)** | 행이 아니라 **묶음(군집, 여기서는 기업) 단위**로 복원추출 | 같은 기업이 여러 기준월에 반복 등장하므로 기업 단위로 뽑음 |
| **층화(stratification)** | 비교를 **같은 조건의 층 안에서만** 함 | 기준월 × 규모 5분위 × 업종군 |
| **직접 표준화(direct standardization)** | 층별 결과를 **한 집단(신호 기업)의 구성 비율**에 맞춰 합침 | 신호 기업과 같은 구성이었다면의 기대 약화 건수 |
| **교란(confounding)** | 비교 집단의 구성 차이가 결과 차이를 만드는 것 | 신호 기업이 작은 규모에 몰려 있는 것 |
| **위험집합(risk set)** | 어떤 시점에 사건이 일어날 수 있는 상태에 있는 대상 | 그 달에 판정 가능하지만 아직 신호가 없던 기업 |
| **공통 지지(common support)** | 두 집단이 모두 존재하는 구간에서만 비교 | 비교 기업이 5곳 미만인 층은 병합하거나 제외 |
| **백분위수(percentile)** | 값을 줄 세웠을 때 하위 p% 지점의 값 | 감소 기준(하위 5% = −34.6%), 약화 기준(하위 5% = −64.2%) |
| **사분위범위(IQR)** | 25% 지점과 75% 지점의 차이. 흩어진 정도 | 활동량 하한 산출 |
| **홀드아웃(hold-out), 독립 검증** | 규칙을 정하는 데 쓰지 않은 데이터에서 성능 확인 | P35만 1,412곳 |
| **민감도 분석(sensitivity analysis)** | 가정·기준·정의를 바꿔도 결론이 유지되는지 확인 | 20가지 변형 |
| **표준화 차이(standardized difference)** | 두 집단 평균(비율)의 차이를 표준편차로 나눈 값. 0.1 이하면 균형 | 층화·표준화 전후 균형 점검 |
| **평균 회귀(regression to the mean)** | 일시적으로 높거나 낮았던 값이 평소 수준으로 돌아오는 현상 | 신호와 약화가 함께 나타나는 이유가 될 수 있음 |
| **적중(precision)과 포착(recall)** | 적중 = 신호 중 실제 약화 비율, 포착 = 약화 중 신호가 있던 비율 | 24.6%, 10.8% |
| **lift(리프트)** | 두 사건이 함께 일어나는 빈도가 우연일 때의 몇 배인지 | 채널 이동 확인(3장) |
| **상관계수(correlation)** | 두 변수가 함께 움직이는 정도(−1~+1) | 채널 간 월별 변동 |
| **주성분분석(PCA)** | 여러 변수를 공통의 큰 흐름 몇 개로 요약 | 채널 구조 확인(3장) |
| **자카드 일치도(Jaccard)** | 두 집합의 교집합 ÷ 합집합 | 신호 기업이 규칙마다 같은지 |
| **결측(missing)과 보간(interpolation)** | 값이 없는 것 / 앞뒤 값으로 채우는 것 | 빈 달은 채우지 않음(M0), 채움은 민감도(M1) |
| **최소 검출 비율비(MDE)** | 주어진 표본 크기로 구별할 수 있는 가장 작은 RR | 신호 기업 488곳·비교 2,023곳이면 약 1.8 |
| **단측 p값** | '우연히 이만큼 나올 확률'을 한쪽 방향으로만 본 값 | 부트스트랩 2,000회 중 RR ≤ 1이 없어 약 0.0005(최소 가능값) |
| **순서형(ordinal) 자료** | 크기 순서만 있고 간격은 고르지 않은 자료 | 건수 구간 코드 0~9 |

---
# 부록 B. 이 보고서의 근거 — 노트북·코드 대응표와 재현 방법

| 장 | 근거 노트북 (`notebooks/`) | 핵심 코드 (`code/`) |
|---|---|---|
| 2장 데이터 | `01_표본정의_패널구축`, `02_건수수치화` | `cntlib.py` |
| 3장 채널 구조 | `03_채널구조탐색` | `cntlib.py` |
| 4장 신호 정의 | `04_활동위축_탐지규칙` | `detect_cnt.py` |
| 6장 핵심 결과 | `05_입금검증` | `validate_cnt.py` |
| 7장 고객별 차이 | `05_입금검증`, `06_채널유형_위축비교` | `validate_cnt.py`, `types_cnt.py` |
| 8장 민감도 | `07_민감도` | `robust_cnt.py` |
| 9장 보강 점검 | `08_보강검증` | `hit_cnt.py` |

**이 노트북이 곧 재현 방법임**

1. 맨 위 **설정 셀**의 `DATA_PATH`에 **레포 밖** 원본 경로를 넣고(또는 환경변수 `CNT_DATA_PATH`), 위에서부터 실행함. 산출물은 레포의 `data/work_cnt/`(`.gitignore` 대상)에 저장됨.
2. 처음부터 다시 계산하려면 `FORCE_RERUN = True`로 바꿈. 이미 01~08을 돌린 결과가 있으면 기본값(`False`)으로 그 결과를 그대로 읽어 보고서만 다시 그림.
3. **점검표**(0.3)가 전부 `일치`이면 기준 실행(2026-10-02)과 같은 결과임. `차이`가 있으면 원본·설정·시드를 확인함.
4. 코드만 먼저 점검하려면 합성 데이터로 `python tools/smoke_test.py`를 실행함(이 노트북도 포함해서 끝까지 돌려 봄). 합성 결과에는 '합성(가짜) 데이터' 경고가 출력됨.

**데이터 취급**

- 이 노트북의 수치는 **집계값**이며 법인 식별 정보·행 단위 값이 없음. 5 미만 칸은 가려져 있음.
- 실행하면 출력 셀에 집계값과 그림이 저장되고 설정 셀에 로컬 경로가 남음. **커밋하기 전에 `python tools/strip_outputs.py`** 로 지우고(`--check`로 확인), 외부에 공유하기 전에는 **팀의 데이터 반출 검토 절차**를 거쳐야 함.